<a href="https://colab.research.google.com/github/fzampirolli/pdi-vc/blob/master/notebooks_alunos/cpp.it/cap04/cap04.EPs_aluno.ipynb"><img src="imagens/colab-badge.png" style="height:20px;vertical-align:middle"></a>
<a href="https://github.com/fzampirolli/pdi-vc"><img src="imagens/github-badge.png" style="height:20px;vertical-align:middle"></a>

## 💻 **Parte Pratica con Esercizi di Programmazione**


Questa lista trasforma i concetti del Capitolo 4 in un percorso pratico di segmentazione e morfologia matematica. Gli EP iniziano con la sogliatura e avanzano fino all'etichettatura e ai descrittori delle componenti, utilizzando sempre matrici di piccole dimensioni affinché ogni pixel possa essere verificato a mano.

> ### ❗ Regola comune degli EP morfologici
>
> Nelle operazioni con vicinato, **non eseguire il padding**. Per ogni pixel, valutare solo le posizioni dell'elemento strutturante che ricadono all'interno del dominio dell'immagine. Questa è la stessa idea delle implementazioni didattiche in `morph.py`, come `mm::dil0`, `mm::ero0`, `mm::dil1` e `mm::label0`: il vicinato viene ritagliato dal dominio valido dell'immagine.

### 🎯 Obiettivo di questo Quaderno

Il quaderno consente di sviluppare, validare, organizzare e testare soluzioni di **Esercizi di Programmazione (EP)** in ambienti interattivi, come Colab, con gli stessi casi di test di Moodle, copiandoli lì solo al momento di registrare il voto ufficiale.

#### *Download*

Scarica `morph.py` e `testsuite.py` eseguendo la cella qui sotto:

In [ ]:
import os, urllib.request

os.makedirs("tmp/state", exist_ok=True)  # artefatti di build della pista C++ (.cpp, binario, PNG)

url = "https://raw.githubusercontent.com/fzampirolli/pdi-vc/master/morph/config.py"
if not os.path.exists("config.py"):
    urllib.request.urlretrieve(url, "config.py")

# Il kernel è comunque Python anche nella pista C++: `mm` (morph.py) è usato dai
# simulatori, dalla visualizzazione delle figure che il binario C++ genera e dallo
# stato mm::Image tra le celle. cpp=True scarica anche la pista compilata
# (morph.hpp + stb_image*.h), usata nell'#include delle celle %%writefile *.cpp.
import config
config.setup(testsuite=True, cpp=True)
from morph import mm
from testsuite import TestSuite

#### Esecuzione dei Test
Per valutare i test, eseguire `TestSuite("EP04_01.extensão").run()` in una nuova cella, sostituendo l'estensione con quella del linguaggio utilizzato (`.py`, `.java`, `.c`, `.cpp`, `.js` o `.r`). Il sistema scarica i casi di test da GitHub, esegue il programma e calcola automaticamente il voto.

Per testare direttamente codice Python, senza salvare un file, utilizzare `run_code(codigo)` passando il codice come *stringa* in una variabile `codigo`:

```python
codigo = """
from morph import mm
# ... tuo codice qui ...
"""
TestSuite("EP04_01").run_code(codigo)
```

### EP04_01 🎚️ Soglia Globale con Soglia Fissa

Negli **scanner di documenti** e nei **sistemi di lettura di codici a barre**, la prima fase dell'elaborazione consiste sempre nel separare ciò che è "oggetto" (inchiostro, testo, barre) da ciò che è "sfondo" (carta, imballaggio). La **soglia globale** fa esattamente questo: confronta ogni pixel con un'unica soglia $T$ e decide, in tempo reale, se esso appartiene alla classe chiara o alla classe scura. È l'operatore di segmentazione più semplice — eppure è alla base di gran parte dei *pipeline* industriali di ispezione visiva.
Vedi in [Figura 4.1](#fig-04-sim-ep0401-limiar) una simulazione di questo EP.

#### 📋 Linee Guida di Implementazione

1. **Dimensioni:** Leggere gli interi $L$ (righe) e $C$ (colonne).
2. **Soglia:** Leggere l'intero $T$ (soglia di decisione).
3. **Dati:** Leggere i valori interi della matrice originale riga per riga.
4. **Mappatura:** Per ogni pixel $p$, calcolare il nuovo valore tramite l'equazione:

$$
p' =
\begin{cases}
255, & \text{se } p > T \\
0, & \text{se } p \le T
\end{cases}
$$
5. **Output:** Visualizzare la matrice binarizzata con dimensioni $L \times C$.

#### 📌 Vincoli Computazionali

* **Binarizzazione:** L'output contiene **solo** i valori $0$ o $255$.
* **Confronto rigoroso:** Il criterio usa $> T$ (i pixel uguali a $T$ diventano sfondo).
* **Tipo:** Il risultato finale deve essere intero.
* **Osservazione:** Questo EP segue la convenzione di OpenCV (`cv2.THRESH_BINARY`): solo i pixel con valore **maggiore di** $T$ diventano bianchi (`255`); i pixel con valore **uguale a** $T$ restano neri (`0`).

#### 🧠 Fondamenti Teorici

| Parametro | Tipo | Impatto Visivo |
|-----------|------|----------------|
| **$T$ piccolo** | Intero | La maggior parte dei pixel diventa bianca |
| **$T$ grande**  | Intero | La maggior parte dei pixel diventa nera |
| **$T$ ben scelto** | Intero | Separa nettamente oggetto e sfondo |

#### 📦 Specifica di Input e Output (VPL)

**Input:**

* Riga 1: Intero $L$.
* Riga 2: Intero $C$.
* Riga 3: Intero $T$.
* Righe successive: Elementi interi della matrice originale.

**Output:**

* Matrice binarizzata in $L$ righe e $C$ colonne, valori $0$ o $255$ separati da spazi.

#### 📌 Esempi

| Input | Output | Osservazione |
|---------|-------|------------|
| 2<br>4<br>100<br>0 99 100 180<br>255 30 120 80 | 0 0 0 255<br>255 0 255 0 | $T=100$: solo i pixel con valore maggiore di 100 diventano bianchi; <br>pertanto, 99 e 100 diventano neri. |
| 1<br>3<br>0<br>0 50 255 | 0 255 255 | $T=0$: solo i pixel con valore strettamente maggiore di 0 diventano bianchi. |

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0401-limiar" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

  <!-- Cabeçalho -->
  <div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241d;">🎚️ Simulatore EP04_01: Soglia Globale</span>
    <span style="background:#26241d;color:#7ee7c6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;font-family:monospace;">p' = (p > T) ? 255 : 0</span>
  </div>

  <div style="padding:16px;background:#ffffff;">
    <p style="font-size:11px;color:#777;margin-bottom:12px;text-align:center;">👆 Clicca su una cella della <b>Ingresso Originale</b> per scurire il pixel (−30) e clicca con il tasto destro per schiarire (+30). Regola la soglia T per la binarizzazione.</p>

    <!-- Controle do Limiar T -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;margin-bottom:16px;">
      <div style="display:flex;justify-content:space-between;align-items:center;margin-bottom:6px;">
        <label style="font-size:11px;font-weight:700;color:#2980b9;">T (Soglia)</label>
        <span id="sim_ep0401_vl_t" style="font-family:monospace;font-size:12px;font-weight:700;color:#2980b9;">128</span>
      </div>
      <input type="range" id="sim_ep0401_sl_t" min="0" max="255" step="1" value="128" style="width:100%;cursor:pointer;">
    </div>

    <!-- Comparativo Lado a Lado: Entrada vs Resultado Binarizado -->
    <div style="display:grid;grid-template-columns:repeat(auto-fit, minmax(200px, 1fr));gap:16px;align-items:start;margin-bottom:14px;">
      
      <!-- Entrada Original -->
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#27ae60;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">Ingresso Originale (Cliccabile)</span>
        <div id="sim_ep0401_grid_orig" style="display:grid;grid-template-columns:repeat(4, 42px);gap:4px;justify-content:center;user-select:none;"></div>
        <button id="sim_ep0401_btnNew" style="margin-top:12px;padding:6px 12px;font-size:11px;font-weight:700;border:1px solid #26241d;background:#26241d;color:#7ee7c6;cursor:pointer;border-radius:9px;white-space:nowrap;font-family:monospace;transition:all 0.15s ease;">🎲 Nuova Immagine</button>
      </div>

      <!-- Resultado Binarizado -->
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#2980b9;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">Risultato Binarizzato (p')</span>
        <div id="sim_ep0401_grid_new" style="display:grid;grid-template-columns:repeat(4, 42px);gap:4px;justify-content:center;user-select:none;"></div>
        <button id="sim_ep0401_btnReset" style="margin-top:12px;padding:6px 12px;font-size:11px;font-weight:600;border:1px solid #e4dcc8;background:#f1ead7;color:#5e5a4a;cursor:pointer;border-radius:9px;white-space:nowrap;transition:all 0.15s ease;">↩ Ripristina Soglia (T = 128)</button>
      </div>

    </div>

    <!-- Painel Explicativo Dinâmico -->
    <div id="sim_ep0401_debug" style="background:#f1ead7;border:1px solid #e4dcc8;border-radius:8px;padding:8px 12px;text-align:center;font-size:11px;font-family:monospace;color:#26241d;">
      Formula applicata: <b>(p > 128) ? 255 : 0</b>
    </div>

  </div>
</div>

<script>
(function(){
  function initSimEP0401(root){
    if (!root || root.dataset.simEp0401Init) return;
    root.dataset.simEp0401Init = "1";

    var slT      = root.querySelector('#sim_ep0401_sl_t');
    var vlT      = root.querySelector('#sim_ep0401_vl_t');
    var gridOrig = root.querySelector('#sim_ep0401_grid_orig');
    var gridNew  = root.querySelector('#sim_ep0401_grid_new');
    var debugDiv = root.querySelector('#sim_ep0401_debug');

    var btnNew   = root.querySelector('#sim_ep0401_btnNew');
    var btnReset = root.querySelector('#sim_ep0401_btnReset');

    var pixels = Array(16).fill(0).map(function(){ return Math.floor(Math.random() * 256); });

    function renderOrig() {
      gridOrig.innerHTML = '';
      pixels.forEach(function(p, idx) {
        var cellO = document.createElement('div');
        var fgColorO = p > 128 ? '#000000' : '#ffffff';
        cellO.style.cssText = 'width:42px;height:42px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;font-family:monospace;border-radius:6px;border:1px solid #e4dcc8;user-select:none;cursor:pointer;background:rgb(' + p + ',' + p + ',' + p + ');color:' + fgColorO + ';box-sizing:border-box;';
        cellO.textContent = p;
        cellO.title = 'Clique esquerdo: escurece (-30) | Botão direito: clareia (+30)';

        cellO.addEventListener('click', function(e) {
          e.preventDefault();
          pixels[idx] = Math.max(0, pixels[idx] - 30);
          render();
        });

        cellO.addEventListener('contextmenu', function(e) {
          e.preventDefault();
          pixels[idx] = Math.min(255, pixels[idx] + 30);
          render();
        });

        gridOrig.appendChild(cellO);
      });
    }

    function render() {
      var T = parseInt(slT.value, 10) || 0;
      vlT.textContent = T;
      debugDiv.innerHTML = 'Fórmula aplicada: <b>(p > ' + T + ') ? 255 : 0</b>';

      renderOrig();
      gridNew.innerHTML = '';

      pixels.forEach(function(p) {
        var res = (p > T) ? 255 : 0;
        var fgColorN = res > 128 ? '#000000' : '#ffffff';
        var cellN = document.createElement('div');
        cellN.style.cssText = 'width:42px;height:42px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;font-family:monospace;border-radius:6px;border:1px solid #e4dcc8;user-select:none;transition:all 0.15s ease;background:rgb(' + res + ',' + res + ',' + res + ');color:' + fgColorN + ';box-sizing:border-box;';
        cellN.textContent = res;
        gridNew.appendChild(cellN);
      });
    }

    slT.addEventListener('input', render);

    btnNew.addEventListener('click', function() {
      pixels = Array(16).fill(0).map(function(){ return Math.floor(Math.random() * 256); });
      render();
    });

    btnReset.addEventListener('click', function() {
      slT.value = '128';
      render();
    });

    render();
  }

  function tryInitSimEP0401(){
    var root = document.getElementById('sim-ep0401-limiar');
    if (root) initSimEP0401(root); else setTimeout(tryInitSimEP0401, 200);
  }
  tryInitSimEP0401();
})();
</script>
</div>
""")

**Figura 4.1:** Simulatore EP04_01: Soglia Globale con Soglia Fissa (p


<figure id="fig-04-sim-ep0401-limiar">
  <img src="imagens/fig-04-sim-ep0401-limiar.png" alt=" Simulatore EP04_01: Soglia Globale con Soglia Fissa (p' = (p > T) ? 255 : 0) " style="max-width:80%" />
  <figcaption><strong>Figura 4.1:</strong>  Simulatore EP04_01: Soglia Globale con Soglia Fissa (p' = (p > T) ? 255 : 0) </figcaption>
</figure>

In [ ]:
%%writefile EP04_01.cpp
// your solution

In [ ]:
TestSuite("EP04_01.cpp").run()

### EP04_02 📊 Soglia Automatica di Otsu

Scegliere manualmente la soglia $T$ funziona quando l'illuminazione è stabile, ma in **microscopia digitale** e nell'**ispezione di strisci di sangue**, ogni campione ha un contrasto diverso — una soglia fissa fallirebbe da un'immagine all'altra. Il **metodo di Otsu** risolve questo problema trovando, da solo, la soglia che **massimizza la separazione statistica** tra le due classi di pixel, rendendo la segmentazione automatica e adattiva.
Vedi in [Figura 4.2](#fig-04-sim-ep0402-otsu) una simulazione di questo EP.

#### 📋 Linee Guida di Implementazione

1. **Dimensioni:** Leggere gli interi $L$ (righe) e $C$ (colonne).
2. **Dati:** Leggere i valori interi della matrice originale riga per riga.
3. **Istogramma:** Costruire l'istogramma $h[i]$, $i=0,\dots,255$, contando quanti pixel hanno valore $i$.
4. **Ricerca della soglia:** Per ogni candidato $T$ da $1$ a $255$, calcolare la **varianza tra le classi**:
$$
\sigma_B^2(T) = \frac{n_0 \cdot n_1}{N^2}\,(m_0 - m_1)^2
$$
dove $n_0,n_1$ sono le quantità di pixel con valore $<T$ e $\geq T$, $m_0,m_1$ sono le loro medie, e $N=L\times C$.

5. **Scelta:** La soglia ottimale $T^*$ è quella che massimizza $\sigma_B^2(T)$ (in caso di pareggio, mantenere la **prima** trovata).
6. Applicazione: Binarizzare l'immagine usando T*, applicando:
$$
p' =
\begin{cases}
255, & \text{se } p > T^* \\
0, & \text{se } p \le T^*
\end{cases}
$$

#### 📌 Vincoli Computazionali

* **Candidati validi:** Ignorare $T$ che lasci $n_0=0$ o $n_1=0$ (classe vuota).
* **Pareggio:** Mantenere sempre la **prima** $T$ che ha raggiunto il valore massimo di $\sigma_B^2$.
* **Tipo:** $T^*$ e la matrice di uscita devono essere interi.
* **Convenzione OpenCV:** La binarizzazione segue `cv2.THRESL_BINARY`; i pixel con valore esattamente uguale a $T^*$ diventano neri.

#### 🧠 Fondamenti Teorici

| Concetto | Significato | Impatto |
|----------|-------------|---------|
| **$\sigma_B^2(T)$ alta** | Classi ben separate in $T$ | $T$ è un buon candidato come soglia |
| **Istogramma bimodale** | Due "colline" distinte | Otsu trova la valle tra di esse |
| **Istogramma unimodale** | Una sola "collina" | Otsu sceglie comunque *qualche* $T$, ma la segmentazione è poco affidabile |

#### 📦 Specifica di Input e Output (VPL)

**Input:**

* Riga 1: Intero $L$.
* Riga 2: Intero $C$.
* Righe successive: Elementi interi della matrice originale.

**Output:**

* Matrice binarizzata in $L$ righe e $C$ colonne, valori $0$ o $255$.

#### 📌 Esempi

| Input | Output | Osservazione |
|---------|-------|------------|
| 4<br>4<br>12 12 12 200<br>12 12 200 200<br>12 200 200 200<br>200 200 200 200 | 0 0 0 255<br>0 0 255 255<br>0 255 255 255<br>255 255 255 255 | Istogramma bimodale chiaro: 12 e 200 |
| 1<br>2<br>10 250 | 0 250 | Solo due valori: $T^*$ si colloca sul maggiore |

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0402-otsu" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

  <!-- Cabeçalho -->
  <div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241d;">📊 Simulatore EP04_02: Otsu Automatico</span>
    <span style="background:#26241d;color:#7ee7c6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;font-family:monospace;">T* = argmax σ²_B(T)</span>
  </div>

  <div style="padding:16px;background:#ffffff;">
    <p style="font-size:11px;color:#777;margin-bottom:12px;text-align:center;">👆 Clic sinistro scurisce (−25) e clic con tasto destro schiarisce (+25) i pixel dell'input. Osserva la soglia ottimale T* adattarsi dinamicamente all'istogramma.</p>

    <!-- Painel do Histograma e T* -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:14px;margin-bottom:16px;">
      <div id="sim_ep0402_hist" style="display:flex;align-items:flex-end;gap:2px;height:100px;margin-bottom:8px;border-bottom:1px solid #e4dcc8;padding-bottom:2px;"></div>
      <p id="sim_ep0402_info" style="text-align:center;font-size:11.5px;font-family:monospace;font-weight:700;color:#26241d;margin:0;">T* = −</p>
    </div>

    <!-- Comparativo Lado a Lado: Entrada Clicável vs Resultado Otsu -->
    <div style="display:grid;grid-template-columns:repeat(auto-fit, minmax(200px, 1fr));gap:16px;align-items:start;margin-bottom:14px;">
      
      <!-- Entrada Original -->
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#27ae60;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">Input Originale (Cliccabile)</span>
        <div id="sim_ep0402_grid_orig" style="display:grid;grid-template-columns:repeat(4, 42px);gap:4px;justify-content:center;user-select:none;"></div>
      </div>

      <!-- Resultado Otsu -->
      <div style="background:#fafaf7;border:1px solid #16a085;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#16a085;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">Risultato Otsu (p')</span>
        <div id="sim_ep0402_grid_new" style="display:grid;grid-template-columns:repeat(4, 42px);gap:4px;justify-content:center;user-select:none;"></div>
      </div>

    </div>

    <!-- Botão de Nova Imagem -->
    <div style="text-align:center;">
      <button id="sim_ep0402_btnNew" style="padding:6px 14px;font-size:11px;font-weight:700;border:1px solid #26241d;background:#26241d;color:#7ee7c6;cursor:pointer;border-radius:9px;white-space:nowrap;font-family:monospace;transition:all 0.15s ease;">🎲 Nuova Immagine (Due Gruppi)</button>
    </div>

  </div>
</div>

<script>
(function(){
  function initSimEP0402(root){
    if (!root || root.dataset.simEp0402Init) return;
    root.dataset.simEp0402Init = "1";

    var gridOrig = root.querySelector('#sim_ep0402_grid_orig');
    var gridNew  = root.querySelector('#sim_ep0402_grid_new');
    var info     = root.querySelector('#sim_ep0402_info');
    var hist     = root.querySelector('#sim_ep0402_hist');
    var btnNew   = root.querySelector('#sim_ep0402_btnNew');

    var pixels = [];

    function generate() {
      var c1 = 20 + Math.floor(Math.random() * 40);
      var c2 = 180 + Math.floor(Math.random() * 60);
      pixels = [];
      for (var i = 0; i < 16; i++) {
        var base = (Math.random() < 0.5) ? c1 : c2;
        pixels.push(Math.max(0, Math.min(255, base + Math.floor(Math.random() * 16 - 8))));
      }
    }

    function otsu(pix) {
      var histArr = new Array(256).fill(0);
      pix.forEach(function(p){ histArr[p]++; });
      var N = pix.length, bestVar = -1, bestT = 0;
      var total = pix.reduce(function(a, b){ return a + b; }, 0);

      for (var T = 1; T < 256; T++) {
        var n0 = 0, s0 = 0;
        for (var i = 0; i < T; i++) {
          n0 += histArr[i];
          s0 += i * histArr[i];
        }
        var n1 = N - n0, s1 = total - s0;
        if (n0 === 0 || n1 === 0) continue;
        var m0 = s0 / n0, m1 = s1 / n1;
        var v = (n0 * n1) * (m0 - m1) * (m0 - m1) / (N * N);
        if (v > bestVar) {
          bestVar = v;
          bestT = T;
        }
      }
      return bestT;
    }

    function renderOrig() {
      gridOrig.innerHTML = '';
      pixels.forEach(function(p, idx) {
        var cellO = document.createElement('div');
        var fgColorO = p > 128 ? '#000000' : '#ffffff';
        cellO.style.cssText = 'width:42px;height:42px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;font-family:monospace;border-radius:6px;border:1px solid #e4dcc8;user-select:none;cursor:pointer;background:rgb(' + p + ',' + p + ',' + p + ');color:' + fgColorO + ';box-sizing:border-box;';
        cellO.textContent = p;
        cellO.title = 'Clique esquerdo: escurece (-25) | Botão direito: clareia (+25)';

        cellO.addEventListener('click', function(e) {
          e.preventDefault();
          pixels[idx] = Math.max(0, pixels[idx] - 25);
          render();
        });

        cellO.addEventListener('contextmenu', function(e) {
          e.preventDefault();
          pixels[idx] = Math.min(255, pixels[idx] + 25);
          render();
        });

        gridOrig.appendChild(cellO);
      });
    }

    function renderHist(T) {
      hist.innerHTML = '';
      var histArr = new Array(256).fill(0);
      pixels.forEach(function(p){ histArr[p]++; });
      var maxH = Math.max.apply(null, histArr);

      for (var i = 0; i < 256; i += 4) {
        var h = (histArr[i] / (maxH || 1)) * 100;
        var bar = document.createElement('div');
        var col = (i >= T) ? '#2980b9' : '#8a8371';
        bar.style.cssText = 'flex:1;height:' + h + '%;background:' + col + ';border-radius:2px 2px 0 0;';
        hist.appendChild(bar);
      }
    }

    function render() {
      var T = otsu(pixels);
      info.innerHTML = 'T* encontrado = <b>' + T + '</b>';
      renderOrig();
      renderHist(T);

      gridNew.innerHTML = '';
      pixels.forEach(function(p) {
        var res = (p > T) ? 255 : 0;
        var fgColorN = res > 128 ? '#000000' : '#ffffff';
        var cellN = document.createElement('div');
        cellN.style.cssText = 'width:42px;height:42px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;font-family:monospace;border-radius:6px;border:1px solid #e4dcc8;user-select:none;transition:all 0.15s ease;background:rgb(' + res + ',' + res + ',' + res + ');color:' + fgColorN + ';box-sizing:border-box;';
        cellN.textContent = res;
        gridNew.appendChild(cellN);
      });
    }

    btnNew.addEventListener('click', function(){
      generate();
      render();
    });

    generate();
    render();
  }

  function tryInitSimEP0402(){
    var root = document.getElementById('sim-ep0402-otsu');
    if (root) initSimEP0402(root); else setTimeout(tryInitSimEP0402, 200);
  }
  tryInitSimEP0402();
})();
</script>
</div>
""")

**Figura 4.2:** Simulatore EP04_02: Sogliatura Automatica di Otsu (T* = argmax σ²_B(T))


<figure id="fig-04-sim-ep0402-otsu">
  <img src="imagens/fig-04-sim-ep0402-otsu.png" alt=" Simulatore EP04_02: Sogliatura Automatica di Otsu (T* = argmax σ²_B(T)) " style="max-width:80%" />
  <figcaption><strong>Figura 4.2:</strong>  Simulatore EP04_02: Sogliatura Automatica di Otsu (T* = argmax σ²_B(T)) </figcaption>
</figure>

In [ ]:
%%writefile EP04_02.cpp
// your solution

In [ ]:
TestSuite("EP04_02.cpp").run()

### EP04_03 🌱 Dilatazione Binaria Piana (mm.dil0)

Nella **microscopia di particelle** e nell'**OCR di targhe automobilistiche usurate**, tratti sottili o discontinui devono essere "ingrossati" affinché il riconoscimento funzioni. La **dilatazione morfologica** fa esattamente questo: espande le regioni chiare utilizzando un elemento strutturante $B$ — la stessa operazione implementata in `morph.py` come `mm::dil0(f, B)`, utilizzata quando $B$ è **piano** (senza pesi, solo $0$/$1$).
Vedi in [Figura 4.3](#fig-04-sim-ep0403-dilatacao) una simulazione di questo EP.

#### 📋 Linee Guida di Implementazione

1. **Dimensioni dell'immagine:** Leggere gli interi $L$ (righe) e $C$ (colonne) di $f$.
2. **Dimensioni di $B$:** Leggere gli interi $L_B$ (righe) e $C_B$ (colonne) dell'elemento strutturante.
3. **Elemento strutturante:** Leggere la matrice $B$ con valori $0$ o $1$, riga per riga.
4. **Dati:** Leggere la matrice $f$ (l'immagine originale), riga per riga.
5. **Riflessione:** Costruire $B_{ref}$, la versione di $B$ riflessa di $180°$ (righe e colonne invertite) — esattamente come fa `mm::dil0` internamente.
6. **Vicinato senza padding:** Per ogni pixel $(y,x)$, percorrere le posizioni $(by,bx)$ di $B_{ref}$ centrate su $(y,x)$, usando lo spostamento
$$
v_y = y + by + o_y,\quad v_x = x + bx + o_x,\quad o_y=-\tfrac{L_B}{2}+0{,}5,\quad o_x=-\tfrac{C_B}{2}+0{,}5
$$
**Scartare** ogni $(v_y,v_x)$ al di fuori di $[0,L)\times[0,C)$ — **non riempire con zeri**.
7. **Mappatura:** Calcolare ogni pixel di uscita come il **massimo** tra $f(y,x)$ e tutti gli $f(v_y,v_x)$ validi la cui posizione corrispondente in $B_{ref}$ vale $1$:
$$
g(y,x) = \max\Big(f(y,x),\ \max_{\substack{(v_y,v_x)\ \text{valido}\\ B_{ref}(by,bx)=1}} f(v_y,v_x)\Big)
$$
8. **Uscita:** Visualizzare la matrice $g$ con dimensioni $L \times C$.

#### 📌 Vincoli Computazionali

* **Senza padding:** Non inventare mai vicini al di fuori dell'immagine; utilizzare solo quelli che esistono realmente.
* **Riflessione obbligatoria:** $B$ deve essere riflesso prima dell'applicazione (è ciò che distingue `mm::dil0` da una semplice ricerca del massimo).
* **Robustezza ai bordi:** Se nessuna posizione valida di $B_{ref}=1$ ricade all'interno del dominio per un dato pixel, questo **mantiene il suo valore originale**.

#### 🧠 Fondamento Teorico

| Concetto | Significato | Impatto Visivo |
|----------|-------------|-----------------|
| **Dilatazione** | $g \geq f$ sempre (estensiva) | Le regioni chiare crescono, i buchi scuri si restringono |
| **$B$ più grande** | Vicinato più ampio | Crescita più aggressiva |
| **Riflessione di $B$** | $B_{ref}(y,x) = B(-y,-x)$ | Garantisce la definizione formale di Minkowski della dilatazione |

#### 📦 Specifica di Input e Output (VPL)

**Input:**

* Riga 1: Intero $L$.
* Riga 2: Intero $C$.
* Riga 3: Intero $L_B$.
* Riga 4: Intero $C_B$.
* Successive $L_B$ righe: elementi interi ($0$ o $1$) della matrice $B$.
* Successive $L$ righe: elementi interi della matrice $f$.

**Output:**

* Matrice $g$ in $L$ righe e $C$ colonne, valori interi separati da spazio.

#### 📌 Esempi

| Input | Output | Osservazione |
|---------|-------|------------|
| 3<br>3<br>3<br>3<br>0 1 0<br>1 1 1<br>0 1 0<br>0 0 0<br>0 9 0<br>0 0 0 | 0 9 0<br>9 9 9<br>0 9 0 | $B$ a croce simmetrico: punto isolato si espande a croce |
| 1<br>4<br>1<br>3<br>1 1 1<br>10 200 5 80 | 200 200 200 80 | $B$ orizzontale: ogni pixel "attira" il massimo dei vicini della riga |

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0403-dilatacao" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

  <!-- Cabeçalho -->
  <div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241d;">🌱 Simulatore EP04_03: Dilatazione Piana (mm.dil0)</span>
    <span style="background:#26241d;color:#7ee7c6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;font-family:monospace;">g = f ⊕ B</span>
  </div>

  <div style="padding:16px;background:#ffffff;">
    <p style="font-size:11px;color:#777;margin-bottom:12px;text-align:center;">Cambia l'elemento strutturante B (o seleziona i preset) e clicca sulle celle dell'immagine originale f per accendere o spegnere i pixel.</p>

    <!-- Painel do Elemento Estruturante B -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:14px;margin-bottom:16px;text-align:center;">
      <span style="font-size:10px;font-weight:700;color:#16a085;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:8px;">Elemento Strutturante B (Clicca per Alternare 0/1)</span>
      <div id="sim_ep0403_grid_B" style="display:grid;grid-template-columns:repeat(3, 38px);gap:4px;justify-content:center;margin-bottom:12px;user-select:none;"></div>
      
      <!-- Presets de B -->
      <div style="display:flex;gap:8px;justify-content:center;flex-wrap:wrap;">
        <button id="sim_ep0403_btnCross" style="padding:5px 10px;font-size:11px;font-weight:600;border:1px solid #e4dcc8;background:#f1ead7;color:#5e5a4a;cursor:pointer;border-radius:8px;transition:all 0.15s ease;">➕ Croce</button>
        <button id="sim_ep0403_btnBox" style="padding:5px 10px;font-size:11px;font-weight:600;border:1px solid #e4dcc8;background:#f1ead7;color:#5e5a4a;cursor:pointer;border-radius:8px;transition:all 0.15s ease;">⬛ Quadrato</button>
        <button id="sim_ep0403_btnDiag" style="padding:5px 10px;font-size:11px;font-weight:600;border:1px solid #e4dcc8;background:#f1ead7;color:#5e5a4a;cursor:pointer;border-radius:8px;transition:all 0.15s ease;">⤫ Diagonale</button>
      </div>
    </div>

    <!-- Comparativo Lado a Lado: Entrada Original f vs Dilatada g -->
    <div style="display:grid;grid-template-columns:repeat(auto-fit, minmax(200px, 1fr));gap:16px;align-items:start;margin-bottom:14px;">
      
      <!-- Imagem Original f -->
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#27ae60;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">Immagine Originale f (5×5)</span>
        <div id="sim_ep0403_grid_orig" style="display:grid;grid-template-columns:repeat(5, 36px);gap:3px;justify-content:center;user-select:none;"></div>
        <button id="sim_ep0403_btnNew" style="margin-top:12px;padding:6px 12px;font-size:11px;font-weight:700;border:1px solid #26241d;background:#26241d;color:#7ee7c6;cursor:pointer;border-radius:9px;white-space:nowrap;font-family:monospace;transition:all 0.15s ease;">🎲 Nuova Immagine</button>
      </div>

      <!-- Imagem Dilatada g -->
      <div style="background:#fafaf7;border:1px solid #16a085;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#16a085;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">Dilatata g (f ⊕ B)</span>
        <div id="sim_ep0403_grid_new" style="display:grid;grid-template-columns:repeat(5, 36px);gap:3px;justify-content:center;user-select:none;"></div>
        <div style="margin-top:12px;padding:6px 12px;font-size:11px;font-weight:600;border:1px solid transparent;background:transparent;color:transparent;user-select:none;">&nbsp;</div>
      </div>

    </div>

    <!-- Painel Explicativo Dinâmico -->
    <div id="sim_ep0403_debug" style="background:#f1ead7;border:1px solid #e4dcc8;border-radius:8px;padding:8px 12px;text-align:center;font-size:11px;font-family:monospace;color:#26241d;">
      g(y,x) = max sui vicini validi di B riflesso
    </div>

  </div>
</div>

<script>
(function(){
  function initSimEP0403(root){
    if (!root || root.dataset.simEp0403Init) return;
    root.dataset.simEp0403Init = "1";

    var gB      = root.querySelector('#sim_ep0403_grid_B');
    var gO      = root.querySelector('#sim_ep0403_grid_orig');
    var gN      = root.querySelector('#sim_ep0403_grid_new');
    var debugDiv= root.querySelector('#sim_ep0403_debug');

    var btnNew   = root.querySelector('#sim_ep0403_btnNew');
    var btnCross = root.querySelector('#sim_ep0403_btnCross');
    var btnBox   = root.querySelector('#sim_ep0403_btnBox');
    var btnDiag  = root.querySelector('#sim_ep0403_btnDiag');

    var B = [[0, 1, 0], [1, 1, 1], [0, 1, 0]];
    var L = 5, C = 5, pixels = [];

    function generate() {
      pixels = [];
      for (var y = 0; y < L; y++) {
        var row = [];
        for (var x = 0; x < C; x++) {
          row.push((Math.random() < 0.78) ? 0 : 1);
        }
        pixels.push(row);
      }
    }

    function reflect(M) {
      var n = M.length, out = [];
      for (var i = n - 1; i >= 0; i--) {
        out.push(M[i].slice().reverse());
      }
      return out;
    }

    function dilate(f, Bm) {
      var HB = Bm.length, WB = Bm[0].length, oy = -HB / 2 + 0.5, ox = -WB / 2 + 0.5;
      var Bref = reflect(Bm);
      var g = [];
      for (var y = 0; y < L; y++) g.push(f[y].slice());

      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) {
          for (var by = 0; by < HB; by++) {
            for (var bx = 0; bx < WB; bx++) {
              if (Bref[by][bx] !== 1) continue;
              var vy = Math.trunc(y + by + oy), vx = Math.trunc(x + bx + ox);
              if (vy >= 0 && vy < L && vx >= 0 && vx < C && f[vy][vx] > g[y][x]) {
                g[y][x] = f[vy][vx];
              }
            }
          }
        }
      }
      return g;
    }

    function renderB() {
      gB.innerHTML = '';
      for (var by = 0; by < 3; by++) {
        for (var bx = 0; bx < 3; bx++) {
          (function(byy, bxx){
            var c = document.createElement('div');
            c.style.cssText = 'width:38px;height:38px;display:flex;align-items:center;justify-content:center;font-size:12px;font-weight:700;font-family:monospace;border-radius:6px;cursor:pointer;border:1px solid #e4dcc8;user-select:none;transition:all 0.15s ease;';
            c.style.background = B[byy][bxx] ? '#16a085' : '#fafaf7';
            c.style.color = B[byy][bxx] ? '#ffffff' : '#8a8371';
            c.textContent = B[byy][bxx];

            c.addEventListener('click', function(){
              B[byy][bxx] = 1 - B[byy][bxx];
              renderB();
              render();
            });
            gB.appendChild(c);
          })(by, bx);
        }
      }
    }

    function render() {
      var g = dilate(pixels, B);
      gO.innerHTML = '';
      gN.innerHTML = '';

      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) {
          (function(yy, xx){
            var p = pixels[yy][xx] ? 255 : 30;
            var fgO = p > 128 ? '#000000' : '#ffffff';
            var cO = document.createElement('div');
            cO.style.cssText = 'width:36px;height:36px;display:flex;align-items:center;justify-content:center;font-size:9.5px;font-weight:700;font-family:monospace;border-radius:6px;border:1px solid #e4dcc8;cursor:pointer;user-select:none;background:rgb(' + p + ',' + p + ',' + p + ');color:' + fgO + ';box-sizing:border-box;';
            cO.textContent = pixels[yy][xx];

            cO.addEventListener('click', function(){
              pixels[yy][xx] = 1 - pixels[yy][xx];
              render();
            });
            gO.appendChild(cO);
          })(y, x);

          var r = g[y][x] ? 255 : 30;
          var fgN = r > 128 ? '#000000' : '#ffffff';
          var cN = document.createElement('div');
          cN.style.cssText = 'width:36px;height:36px;display:flex;align-items:center;justify-content:center;font-size:9.5px;font-weight:700;font-family:monospace;border-radius:6px;border:1px solid #e4dcc8;user-select:none;background:rgb(' + r + ',' + r + ',' + r + ');color:' + fgN + ';box-sizing:border-box;';
          cN.textContent = g[y][x];
          gN.appendChild(cN);
        }
      }
    }

    btnNew.addEventListener('click', function(){
      generate();
      render();
    });

    btnCross.addEventListener('click', function(){
      B = [[0, 1, 0], [1, 1, 1], [0, 1, 0]];
      renderB();
      render();
    });

    btnBox.addEventListener('click', function(){
      B = [[1, 1, 1], [1, 1, 1], [1, 1, 1]];
      renderB();
      render();
    });

    btnDiag.addEventListener('click', function(){
      B = [[1, 0, 1], [0, 1, 0], [1, 0, 1]];
      renderB();
      render();
    });

    generate();
    renderB();
    render();
  }

  function tryInitSimEP0403(){
    var root = document.getElementById('sim-ep0403-dilatacao');
    if (root) initSimEP0403(root); else setTimeout(tryInitSimEP0403, 200);
  }
  tryInitSimEP0403();
})();
</script>
</div>
""")

**Figura 4.3:** Simulatore EP04_03: Dilatazione Binaria Piana (g = f ⊕ B)


<figure id="fig-04-sim-ep0403-dilatacao">
  <img src="imagens/fig-04-sim-ep0403-dilatacao.png" alt=" Simulatore EP04_03: Dilatazione Binaria Piana (g = f ⊕ B) " style="max-width:80%" />
  <figcaption><strong>Figura 4.3:</strong>  Simulatore EP04_03: Dilatazione Binaria Piana (g = f ⊕ B) </figcaption>
</figure>

In [ ]:
%%writefile EP04_03.cpp
// your solution

In [ ]:
TestSuite("EP04_03.cpp").run()

### EP04_04 🪨 Erosione Binaria Piatta (mm.ero0)

Se la dilatazione ingrossa, l'**erosione** assottiglia. Nei **sistemi di conteggio cellulare**, viene utilizzata per **separare cellule che si toccano**: "consumando" i bordi di ogni regione, le connessioni sottili tra gli oggetti scompaiono prima ancora che venga effettuato qualsiasi conteggio. In `morph.py`, questa è l'operazione `mm::ero0(f, B)` — il **duale** esatto della dilatazione, e l'unica delle due che **non** riflette l'elemento strutturante.
Vedi nella [Figura 4.4](#fig-04-sim-ep0404-erosao) una simulazione di questo EP.

#### 📋 Linee Guida di Implementazione

1. **Dimensioni dell'immagine:** Leggere gli interi $L$ (righe) e $C$ (colonne) da $f$.
2. **Dimensioni di $B$:** Leggere gli interi $L_B$ (righe) e $C_B$ (colonne) dell'elemento strutturante.
3. **Elemento strutturante:** Leggere la matrice $B$ con valori $0$ o $1$, riga per riga.
4. **Dati:** Leggere la matrice $f$ (l'immagine originale), riga per riga.
5. **Vicinato senza padding (senza riflessione!):** Per ogni pixel $(y,x)$, percorrere le posizioni $(by,bx)$ di $B$ **nell'ordine originale** (senza riflettere), usando lo stesso spostamento dell'EP04_03:
$$
v_y = y + by + o_y,\quad v_x = x + bx + o_x,\quad o_y=-\tfrac{L_B}{2}+0{,}5,\quad o_x=-\tfrac{C_B}{2}+0{,}5
$$

**Scartare** ogni $(v_y,v_x)$ al di fuori di $[0,L)\times[0,C)$.
6. **Mappatura:** Calcolare ogni pixel di uscita come il **minimo** tra $f(y,x)$ e tutti gli $f(v_y,v_x)$ validi la cui posizione corrispondente in $B$ vale $1$:
$$
g(y,x) = \min\Big(f(y,x),\ \min_{\substack{(v_y,v_x)\ \text{valido}\\ B(by,bx)=1}} f(v_y,v_x)\Big)
$$
7. **Uscita:** Visualizzare la matrice $g$ con dimensioni $L \times C$.

#### 📌 Vincoli Computazionali

* **Senza riflessione:** Diversamente dalla dilatazione, $B$ viene utilizzato **esattamente come letto** — riflettere qui sarebbe un errore concettuale grave.
* **Senza padding:** I vicini al di fuori dell'immagine vengono semplicemente ignorati, mai trattati come $0$.
* **Robustezza ai bordi:** Se nessuna posizione valida di $B=1$ ricade all'interno del dominio, il pixel mantiene il suo valore originale.

#### 🧠 Fondamenti Teorici

| Concetto | Significato | Impatto Visivo |
|----------|-------------|-----------------|
| **Erosione** | $g \leq f$ sempre (anti-estensiva) | Le regioni chiare si restringono, il rumore puntuale scompare |
| **Dualità** | $\text{ero}(f,B) = -\text{dil}(-f, B_{ref})$ | Erosione e dilatazione sono "specchi" matematici |
| **$B$ più grande** | Erosione più aggressiva | Gli oggetti sottili scompaiono completamente |

#### 📦 Specifica di Ingresso e Uscita (VPL)

**Ingresso:**

* Riga 1: Intero $L$.
* Riga 2: Intero $C$.
* Riga 3: Intero $L_B$.
* Riga 4: Intero $C_B$.
* Prossime $L_B$ righe: elementi interi ($0$ o $1$) della matrice $B$.
* Prossime $L$ righe: elementi interi della matrice $f$.

**Uscita:**

* Matrice $g$ in $L$ righe e $C$ colonne, valori interi separati da spazio.

#### 📌 Esempi

| Ingresso | Uscita | Osservazione |
|---------|-------|------------|
| 3<br>3<br>3<br>3<br>0 1 0<br>1 1 1<br>0 1 0<br>9 9 9<br>9 0 9<br>9 9 9 | 9 0 9<br>0 0 0<br>9 0 9 | Il "buco" centrale (0) si propaga a croce |
| 1<br>4<br>1<br>3<br>1 1 1<br>10 200 5 80 | 10 5 5 80 | $B$ orizzontale: ogni pixel "attira" il minimo dei vicini della riga |

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0404-erosao" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

  <!-- Cabeçalho -->
  <div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241d;">🪨 Simulatore EP04_04: Erosione Planare (mm.ero0)</span>
    <span style="background:#26241d;color:#7ee7c6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;font-family:monospace;">g = f ⊖ B</span>
  </div>

  <div style="padding:16px;background:#ffffff;">
    <p style="font-size:11px;color:#777;margin-bottom:12px;text-align:center;">Alterna l'elemento strutturante B (o seleziona i preset) e clicca sulle celle dell'immagine originale f per accendere o spegnere i pixel.</p>

    <!-- Painel do Elemento Estruturante B -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:14px;margin-bottom:16px;text-align:center;">
      <span style="font-size:10px;font-weight:700;color:#c0392b;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:8px;">Elemento Strutturante B (Clicca per Alternare 0/1)</span>
      <div id="sim_ep0404_grid_B" style="display:grid;grid-template-columns:repeat(3, 38px);gap:4px;justify-content:center;margin-bottom:12px;user-select:none;"></div>
      
      <!-- Presets de B -->
      <div style="display:flex;gap:8px;justify-content:center;flex-wrap:wrap;">
        <button id="sim_ep0404_btnCross" style="padding:5px 10px;font-size:11px;font-weight:600;border:1px solid #e4dcc8;background:#f1ead7;color:#5e5a4a;cursor:pointer;border-radius:8px;transition:all 0.15s ease;">➕ Croce</button>
        <button id="sim_ep0404_btnBox" style="padding:5px 10px;font-size:11px;font-weight:600;border:1px solid #e4dcc8;background:#f1ead7;color:#5e5a4a;cursor:pointer;border-radius:8px;transition:all 0.15s ease;">⬛ Scatola</button>
        <button id="sim_ep0404_btnDiag" style="padding:5px 10px;font-size:11px;font-weight:600;border:1px solid #e4dcc8;background:#f1ead7;color:#5e5a4a;cursor:pointer;border-radius:8px;transition:all 0.15s ease;">⤫ Diagonale</button>
      </div>
    </div>

    <!-- Comparativo Lado a Lado: Entrada Original f vs Erodida g -->
    <div style="display:grid;grid-template-columns:repeat(auto-fit, minmax(200px, 1fr));gap:16px;align-items:start;margin-bottom:14px;">
      
      <!-- Imagem Original f -->
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#27ae60;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">Immagine Originale f (5×5)</span>
        <div id="sim_ep0404_grid_orig" style="display:grid;grid-template-columns:repeat(5, 36px);gap:3px;justify-content:center;user-select:none;"></div>
        <button id="sim_ep0404_btnNew" style="margin-top:12px;padding:6px 12px;font-size:11px;font-weight:700;border:1px solid #26241d;background:#26241d;color:#7ee7c6;cursor:pointer;border-radius:9px;white-space:nowrap;font-family:monospace;transition:all 0.15s ease;">🎲 Nuova Immagine</button>
      </div>

      <!-- Imagem Erodida g -->
      <div style="background:#fafaf7;border:1px solid #c0392b;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#c0392b;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">Erosa g (f ⊖ B)</span>
        <div id="sim_ep0404_grid_new" style="display:grid;grid-template-columns:repeat(5, 36px);gap:3px;justify-content:center;user-select:none;"></div>
        <div style="margin-top:12px;padding:6px 12px;font-size:11px;font-weight:600;border:1px solid transparent;background:transparent;color:transparent;user-select:none;">&nbsp;</div>
      </div>

    </div>

    <!-- Painel Explicativo Dinâmico -->
    <div id="sim_ep0404_debug" style="background:#f1ead7;border:1px solid #e4dcc8;border-radius:8px;padding:8px 12px;text-align:center;font-size:11px;font-family:monospace;color:#26241d;">
      g(y,x) = min sui vicini validi di B (senza riflettere)
    </div>

  </div>
</div>

<script>
(function(){
  function initSimEP0404(root){
    if (!root || root.dataset.simEp0404Init) return;
    root.dataset.simEp0404Init = "1";

    var gB      = root.querySelector('#sim_ep0404_grid_B');
    var gO      = root.querySelector('#sim_ep0404_grid_orig');
    var gN      = root.querySelector('#sim_ep0404_grid_new');
    var debugDiv= root.querySelector('#sim_ep0404_debug');

    var btnNew   = root.querySelector('#sim_ep0404_btnNew');
    var btnCross = root.querySelector('#sim_ep0404_btnCross');
    var btnBox   = root.querySelector('#sim_ep0404_btnBox');
    var btnDiag  = root.querySelector('#sim_ep0404_btnDiag');

    var B = [[0, 1, 0], [1, 1, 1], [0, 1, 0]];
    var L = 5, C = 5, pixels = [];

    function generate() {
      pixels = [];
      for (var y = 0; y < L; y++) {
        var row = [];
        for (var x = 0; x < C; x++) {
          row.push((Math.random() < 0.78) ? 1 : 0);
        }
        pixels.push(row);
      }
    }

    function erode(f, Bm) {
      var HB = Bm.length, WB = Bm[0].length, oy = -HB / 2 + 0.5, ox = -WB / 2 + 0.5;
      var g = [];
      for (var y = 0; y < L; y++) g.push(f[y].slice());

      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) {
          for (var by = 0; by < HB; by++) {
            for (var bx = 0; bx < WB; bx++) {
              if (Bm[by][bx] !== 1) continue;
              var vy = Math.trunc(y + by + oy), vx = Math.trunc(x + bx + ox);
              if (vy >= 0 && vy < L && vx >= 0 && vx < C && f[vy][vx] < g[y][x]) {
                g[y][x] = f[vy][vx];
              }
            }
          }
        }
      }
      return g;
    }

    function renderB() {
      gB.innerHTML = '';
      for (var by = 0; by < 3; by++) {
        for (var bx = 0; bx < 3; bx++) {
          (function(byy, bxx){
            var c = document.createElement('div');
            c.style.cssText = 'width:38px;height:38px;display:flex;align-items:center;justify-content:center;font-size:12px;font-weight:700;font-family:monospace;border-radius:6px;cursor:pointer;border:1px solid #e4dcc8;user-select:none;transition:all 0.15s ease;';
            c.style.background = B[byy][bxx] ? '#c0392b' : '#fafaf7';
            c.style.color = B[byy][bxx] ? '#ffffff' : '#8a8371';
            c.textContent = B[byy][bxx];

            c.addEventListener('click', function(){
              B[byy][bxx] = 1 - B[byy][bxx];
              renderB();
              render();
            });
            gB.appendChild(c);
          })(by, bx);
        }
      }
    }

    function render() {
      var g = erode(pixels, B);
      gO.innerHTML = '';
      gN.innerHTML = '';

      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) {
          (function(yy, xx){
            var p = pixels[yy][xx] ? 255 : 30;
            var fgO = p > 128 ? '#000000' : '#ffffff';
            var cO = document.createElement('div');
            cO.style.cssText = 'width:36px;height:36px;display:flex;align-items:center;justify-content:center;font-size:9.5px;font-weight:700;font-family:monospace;border-radius:6px;border:1px solid #e4dcc8;cursor:pointer;user-select:none;background:rgb(' + p + ',' + p + ',' + p + ');color:' + fgO + ';box-sizing:border-box;';
            cO.textContent = pixels[yy][xx];

            cO.addEventListener('click', function(){
              pixels[yy][xx] = 1 - pixels[yy][xx];
              render();
            });
            gO.appendChild(cO);
          })(y, x);

          var r = g[y][x] ? 255 : 30;
          var fgN = r > 128 ? '#000000' : '#ffffff';
          var cN = document.createElement('div');
          cN.style.cssText = 'width:36px;height:36px;display:flex;align-items:center;justify-content:center;font-size:9.5px;font-weight:700;font-family:monospace;border-radius:6px;border:1px solid #e4dcc8;user-select:none;background:rgb(' + r + ',' + r + ',' + r + ');color:' + fgN + ';box-sizing:border-box;';
          cN.textContent = g[y][x];
          gN.appendChild(cN);
        }
      }
    }

    btnNew.addEventListener('click', function(){
      generate();
      render();
    });

    btnCross.addEventListener('click', function(){
      B = [[0, 1, 0], [1, 1, 1], [0, 1, 0]];
      renderB();
      render();
    });

    btnBox.addEventListener('click', function(){
      B = [[1, 1, 1], [1, 1, 1], [1, 1, 1]];
      renderB();
      render();
    });

    btnDiag.addEventListener('click', function(){
      B = [[1, 0, 1], [0, 1, 0], [1, 0, 1]];
      renderB();
      render();
    });

    generate();
    renderB();
    render();
  }

  function tryInitSimEP0404(){
    var root = document.getElementById('sim-ep0404-erosao');
    if (root) initSimEP0404(root); else setTimeout(tryInitSimEP0404, 200);
  }
  tryInitSimEP0404();
})();
</script>
</div>
""")

**Figura 4.4:** Simulatore EP04_04: Erosione Binaria Piana (g = f ⊖ B)


<figure id="fig-04-sim-ep0404-erosao">
  <img src="imagens/fig-04-sim-ep0404-erosao.png" alt=" Simulatore EP04_04: Erosione Binaria Piana (g = f ⊖ B) " style="max-width:80%" />
  <figcaption><strong>Figura 4.4:</strong>  Simulatore EP04_04: Erosione Binaria Piana (g = f ⊖ B) </figcaption>
</figure>

In [ ]:
%%writefile EP04_04.cpp
// your solution

In [ ]:
TestSuite("EP04_04.cpp").run()

### EP04_05 🧹 Apertura Morfologica (Rimozione del Rumore)

Le immagini acquisite da **sensori a basso costo**, come quelli dei droni agricoli, sono spesso disseminate di piccoli punti di rumore — pixel isolati che non rappresentano nulla di reale. Applicare l'erosione seguita dalla dilatazione con lo **stesso** elemento strutturante produce l'**apertura**: essa "pulisce" punti e sottili protuberanze, ma restituisce all'oggetto principale praticamente la sua dimensione originale. È la combinazione classica utilizzata nella **pre-elaborazione di immagini satellitari** prima di qualsiasi conteggio dell'area coltivata.
Vedi in [Figura 4.5](#fig-04-sim-ep0405-abertura) una simulazione di questo EP.

#### 📋 Linee Guida di Implementazione

1. **Dimensioni dell'immagine:** Leggere gli interi $L$ (righe) e $C$ (colonne) da $f$.
2. **Dimensioni di $B$:** Leggere gli interi $L_B$ (righe) e $C_B$ (colonne) dell'elemento strutturante.
3. **Elemento strutturante:** Leggere la matrice $B$ con valori $0$ o $1$, riga per riga.
4. **Dati:** Leggere la matrice binaria $f$ (valori $0$ o $1$), riga per riga.
5. **Erosione:** Calcolare $e = f \ominus B$, usando esattamente l'algoritmo di EP04_04 (senza riflettere $B$, senza padding).
6. **Dilatazione:** Calcolare $g = e \oplus B$, usando esattamente l'algoritmo di EP04_03 (riflettendo $B$, senza padding) — ma ora applicato su $e$, non su $f$.
7. **Output:** Visualizzare la matrice risultante $g$ (l'**apertura** di $f$ tramite $B$) con dimensioni $L \times C$.

#### 📌 Vincoli Computazionali

* **Ordine fisso:** È **sempre** prima erosione, poi dilatazione — l'ordine inverso definisce un altro operatore (chiusura, nel prossimo EP).
* **Stesso $B$:** L'elemento strutturante utilizzato nell'erosione e nella dilatazione deve essere identico.
* **Nessun padding in nessuna delle due fasi.**

#### 🧠 Fondamenti Teorici

| Concetto | Significato | Impatto Visivo |
|----------|-------------|-----------------|
| **Anti-estensività** | $g \subseteq f$ sempre | L'apertura non crea mai nuovi pixel, li rimuove soltanto |
| **Idempotenza** | $\text{apertura}(\text{apertura}(f)) = \text{apertura}(f)$ | Applicarla di nuovo non cambia più nulla |
| **Punti isolati** | Più piccoli di $B$ | Vengono completamente eliminati |
| **Nucleo dell'oggetto** | Più grande di $B$ | Viene recuperato quasi intatto dalla dilatazione finale |

#### 📦 Specifica di Input e Output (VPL)

**Input:**

* Riga 1: Intero $L$.
* Riga 2: Intero $C$.
* Riga 3: Intero $L_B$.
* Riga 4: Intero $C_B$.
* Prossime $L_B$ righe: elementi interi ($0$ o $1$) della matrice $B$.
* Prossime $L$ righe: elementi interi ($0$ o $1$) della matrice $f$.

**Output:**

* Matrice risultante in $L$ righe e $C$ colonne, valori $0$ o $1$.

#### 📌 Esempi

| Input | Output | Osservazione |
|---------|-------|------------|
| 7<br>7<br>3<br>3<br>1 1 1<br>1 1 1<br>1 1 1<br>0 0 0 0 0 0 0<br>0 1 0 0 0 1 0<br>0 0 1 1 1 0 0<br>0 0 1 1 1 0 0<br>0 0 1 1 1 1 0<br>0 0 0 0 0 0 0<br>0 1 0 0 0 0 1 | 0 0 0 0 0 0 0<br>0 0 0 0 0 0 0<br>0 0 1 1 1 0 0<br>0 0 1 1 1 0 0<br>0 0 1 1 1 0 0<br>0 0 0 0 0 0 0<br>0 0 0 0 0 0 0 | Punti isolati e la sottile protuberanza scompaiono; il quadrato centrale sopravvive |

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0405-abertura" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

  <!-- Cabeçalho -->
  <div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241d;">🧹 Simulatore EP04_05: Apertura Morfologica</span>
    <span style="background:#26241d;color:#7ee7c6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;font-family:monospace;">g = (f ⊖ B) ⊕ B</span>
  </div>

  <div style="padding:16px;background:#ffffff;">
    <p style="font-size:11px;color:#777;margin-bottom:12px;text-align:center;">Clicca sulle celle di <b>f originale</b> per accendere o spegnere i pixel (crea il tuo rumore di fondo!) e regola la dimensione dell'elemento strutturante B.</p>

    <!-- Controle do Tamanho de B -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;margin-bottom:16px;text-align:center;">
      <label style="font-size:11px;font-weight:700;color:#8e44ad;">Dimensione di B (Scatola n×n)</label><br>
      <input type="range" id="sim_ep0405_sl_n" min="3" max="5" step="2" value="3" style="width:60%;cursor:pointer;accent-color:#8e44ad;margin-top:6px;">
      <span id="sim_ep0405_vl_n" style="font-family:monospace;font-size:12px;font-weight:700;color:#8e44ad;margin-left:8px;">3×3</span>
    </div>

    <!-- Pipeline em 3 Colunas: f original vs e (Erosão) vs g (Abertura Final) -->
    <div style="display:grid;grid-template-columns:repeat(auto-fit, minmax(180px, 1fr));gap:12px;align-items:start;margin-bottom:14px;">
      
      <!-- f Original -->
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#27ae60;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">f Originale (Cliccabile)</span>
        <div id="sim_ep0405_grid_f" style="display:grid;grid-template-columns:repeat(7, 28px);gap:2px;justify-content:center;user-select:none;"></div>
      </div>

      <!-- e = f ⊖ B -->
      <div style="background:#fafaf7;border:1px solid #c0392b;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#c0392b;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">e = f ⊖ B (Erosione)</span>
        <div id="sim_ep0405_grid_e" style="display:grid;grid-template-columns:repeat(7, 28px);gap:2px;justify-content:center;user-select:none;"></div>
      </div>

      <!-- g = e ⊕ B -->
      <div style="background:#fafaf7;border:2px solid #8e44ad;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#8e44ad;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">g = e ⊕ B (Apertura)</span>
        <div id="sim_ep0405_grid_g" style="display:grid;grid-template-columns:repeat(7, 28px);gap:2px;justify-content:center;user-select:none;"></div>
      </div>

    </div>

    <!-- Botões de Ação -->
    <div style="text-align:center;display:flex;gap:8px;justify-content:center;flex-wrap:wrap;">
      <button id="sim_ep0405_btnNew" style="padding:6px 12px;font-size:11px;font-weight:700;border:1px solid #26241d;background:#26241d;color:#7ee7c6;cursor:pointer;border-radius:9px;white-space:nowrap;font-family:monospace;transition:all 0.15s ease;">🎲 Nuova Immagine (Con Rumore)</button>
      <button id="sim_ep0405_btnClear" style="padding:6px 12px;font-size:11px;font-weight:600;border:1px solid #e4dcc8;background:#f1ead7;color:#5e5a4a;cursor:pointer;border-radius:9px;white-space:nowrap;transition:all 0.15s ease;">🧹 Pulisci Tutto</button>
    </div>

  </div>
</div>

<script>
(function(){
  function initSimEP0405(root){
    if (!root || root.dataset.simEp0405Init) return;
    root.dataset.simEp0405Init = "1";

    var slN     = root.querySelector('#sim_ep0405_sl_n');
    var vlN     = root.querySelector('#sim_ep0405_vl_n');
    var gF      = root.querySelector('#sim_ep0405_grid_f');
    var gE      = root.querySelector('#sim_ep0405_grid_e');
    var gG      = root.querySelector('#sim_ep0405_grid_g');
    var btnNew  = root.querySelector('#sim_ep0405_btnNew');
    var btnClear= root.querySelector('#sim_ep0405_btnClear');

    var L = 7, C = 7, f = [];

    function generate() {
      f = [];
      for (var y = 0; y < L; y++) {
        var row = [];
        for (var x = 0; x < C; x++) row.push(0);
        f.push(row);
      }
      for (var y = 2; y < 5; y++) {
        for (var x = 2; x < 5; x++) f[y][x] = 1;
      }
      for (var k = 0; k < 3; k++) {
        var ry = Math.floor(Math.random() * L), rx = Math.floor(Math.random() * C);
        if (f[ry][rx] === 0 && (ry < 1 || ry > 5 || rx < 1 || rx > 5)) f[ry][rx] = 1;
      }
    }

    function clearAll() {
      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) f[y][x] = 0;
      }
    }

    function box(n) {
      var B = [];
      for (var i = 0; i < n; i++) {
        var row = [];
        for (var j = 0; j < n; j++) row.push(1);
        B.push(row);
      }
      return B;
    }

    function reflect(M) {
      var n = M.length, out = [];
      for (var i = n - 1; i >= 0; i--) out.push(M[i].slice().reverse());
      return out;
    }

    function erode(img, Bm) {
      var HB = Bm.length, WB = Bm[0].length, oy = -HB / 2 + 0.5, ox = -WB / 2 + 0.5;
      var g = [];
      for (var y = 0; y < L; y++) g.push(img[y].slice());

      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) {
          for (var by = 0; by < HB; by++) {
            for (var bx = 0; bx < WB; bx++) {
              if (Bm[by][bx] !== 1) continue;
              var vy = Math.trunc(y + by + oy), vx = Math.trunc(x + bx + ox);
              if (vy >= 0 && vy < L && vx >= 0 && vx < C && img[vy][vx] < g[y][x]) {
                g[y][x] = img[vy][vx];
              }
            }
          }
        }
      }
      return g;
    }

    function dilate(img, Bm) {
      var Bref = reflect(Bm);
      var HB = Bref.length, WB = Bref[0].length, oy = -HB / 2 + 0.5, ox = -WB / 2 + 0.5;
      var g = [];
      for (var y = 0; y < L; y++) g.push(img[y].slice());

      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) {
          for (var by = 0; by < HB; by++) {
            for (var bx = 0; bx < WB; bx++) {
              if (Bref[by][bx] !== 1) continue;
              var vy = Math.trunc(y + by + oy), vx = Math.trunc(x + bx + ox);
              if (vy >= 0 && vy < L && vx >= 0 && vx < C && img[vy][vx] > g[y][x]) {
                g[y][x] = img[vy][vx];
              }
            }
          }
        }
      }
      return g;
    }

    function paintStatic(grid, img) {
      grid.innerHTML = '';
      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) {
          var c = document.createElement('div');
          c.style.cssText = 'width:28px;height:28px;border-radius:4px;border:1px solid #e4dcc8;box-sizing:border-box;';
          c.style.background = img[y][x] === 1 ? '#8e44ad' : '#fafaf7';
          grid.appendChild(c);
        }
      }
    }

    function paintEditable(grid, img) {
      grid.innerHTML = '';
      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) {
          (function(yy, xx){
            var c = document.createElement('div');
            c.style.cssText = 'width:28px;height:28px;border-radius:4px;border:1px solid #e4dcc8;cursor:pointer;box-sizing:border-box;transition:all 0.1s ease;';
            c.style.background = img[yy][xx] === 1 ? '#8e44ad' : '#fafaf7';

            c.addEventListener('click', function(){
              f[yy][xx] = 1 - f[yy][xx];
              render();
            });
            grid.appendChild(c);
          })(y, x);
        }
      }
    }

    function render() {
      var n = parseInt(slN.value, 10) || 3;
      vlN.textContent = n + '×' + n;
      var B = box(n);
      var e = erode(f, B);
      var g = dilate(e, B);

      paintEditable(gF, f);
      paintStatic(gE, e);
      paintStatic(gG, g);
    }

    slN.addEventListener('input', render);

    btnNew.addEventListener('click', function(){
      generate();
      render();
    });

    btnClear.addEventListener('click', function(){
      clearAll();
      render();
    });

    generate();
    render();
  }

  function tryInitSimEP0405(){
    var root = document.getElementById('sim-ep0405-abertura');
    if (root) initSimEP0405(root); else setTimeout(tryInitSimEP0405, 200);
  }
  tryInitSimEP0405();
})();
</script>
</div>
""")

**Figura 4.5:** Simulatore EP04_05: Apertura Morfologica (g = (f ⊖ B) ⊕ B)


<figure id="fig-04-sim-ep0405-abertura">
  <img src="imagens/fig-04-sim-ep0405-abertura.png" alt=" Simulatore EP04_05: Apertura Morfologica (g = (f ⊖ B) ⊕ B) " style="max-width:80%" />
  <figcaption><strong>Figura 4.5:</strong>  Simulatore EP04_05: Apertura Morfologica (g = (f ⊖ B) ⊕ B) </figcaption>
</figure>

In [ ]:
%%writefile EP04_05.cpp
// your solution

In [ ]:
TestSuite("EP04_05.cpp").run()

### EP04_06 🧩 Chiusura Morfologica (Riempimento delle Lacune)

Nella **digitalizzazione delle impronte digitali**, i solchi della pelle talvolta vengono interrotti da sporco o secchezza, creando piccole lacune nella curva continua che dovrebbe esistere. La **chiusura** — dilatazione seguita da erosione con lo stesso elemento strutturante — è l'operatore duale dell'apertura: **riempie piccoli buchi e rientranze strette**, senza alterare significativamente il contorno esterno dell'oggetto. È la fase standard prima di estrarre lo scheletro di un'impronta digitale.
Vedi una simulazione di questo EP in [Figura 4.6](#fig-04-sim-ep0406-fechamento).

#### 📋 Linee Guida di Implementazione

1. **Dimensioni dell'immagine:** Leggere gli interi $L$ (righe) e $C$ (colonne) da $f$.
2. **Dimensioni di $B$:** Leggere gli interi $L_B$ (righe) e $C_B$ (colonne) dell'elemento strutturante.
3. **Elemento strutturante:** Leggere la matrice $B$ con valori $0$ o $1$, riga per riga.
4. **Dati:** Leggere la matrice binaria $f$ (valori $0$ o $1$), riga per riga.
5. **Dilatazione:** Calcolare $d = f \oplus B$, usando esattamente l'algoritmo del EP04_03 (riflettendo $B$, senza padding).
6. **Erosione:** Calcolare $g = d \ominus B$, usando esattamente l'algoritmo del EP04_04 (senza riflettere $B$, senza padding) — ora applicato su $d$, non su $f$.
7. **Uscita:** Visualizzare la matrice risultante $g$ (la **chiusura** di $f$ per $B$) con dimensioni $L \times C$.

#### 📌 Vincoli Computazionali

* **Ordine fisso:** È **sempre** prima la dilatazione, poi l'erosione — l'ordine inverso è l'apertura del EP04_05.
* **Stesso $B$:** L'elemento strutturante usato nella dilatazione e nell'erosione deve essere identico.
* **Nessun padding in nessuna delle due fasi.**

#### 🧠 Fondamento Teorico

| Concetto | Significato | Impatto Visivo |
|----------|-------------|-----------------|
| **Estensività** | $g \supseteq f$ sempre | La chiusura non rimuove mai pixel, solo aggiunge |
| **Idempotenza** | $\text{chiudi}(\text{chiudi}(f)) = \text{chiudi}(f)$ | Applicare di nuovo non cambia più nulla |
| **Piccoli buchi** | Minori di $B$ | Vengono completamente riempiti |
| **Dualità** | $\text{chiudi}(f) = \overline{\text{apri}(\bar f)}$ | È l'apertura applicata al "negativo" dell'immagine |

#### 📦 Specifica di Input e Output (VPL)

**Input:**

* Riga 1: Intero $L$.
* Riga 2: Intero $C$.
* Riga 3: Intero $L_B$.
* Riga 4: Intero $C_B$.
* Prossime $L_B$ righe: elementi interi ($0$ o $1$) della matrice $B$.
* Prossime $L$ righe: elementi interi ($0$ o $1$) della matrice $f$.

**Output:**

* Matrice risultante in $L$ righe e $C$ colonne, valori $0$ o $1$.

#### 📌 Esempi

| Input | Output | Osservazione |
|---------|-------|------------|
| 8<br>8<br>3<br>3<br>1 1 1<br>1 1 1<br>1 1 1<br>0 0 0 0 0 0 0 0<br>0 0 1 1 1 1 0 0<br>0 0 1 1 1 1 0 0<br>0 0 1 0 1 1 0 0<br>0 0 1 1 0 1 0 0<br>0 0 1 1 1 1 0 0<br>0 0 1 1 1 1 0 0<br>0 0 0 0 0 0 0 0 | 0 0 1 1 1 1 0 0<br>0 0 1 1 1 1 0 0<br>0 0 1 1 1 1 0 0<br>0 0 1 1 1 1 0 0<br>0 0 1 1 1 1 0 0<br>0 0 1 1 1 1 0 0<br>0 0 1 1 1 1 0 0<br>0 0 1 1 1 1 0 0 | I due buchi interni non adiacenti vengono completamente riempiti |

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0406-fechamento" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

  <!-- Cabeçalho -->
  <div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241d;">🧩 Simulatore EP04_06: Chiusura Morfologica</span>
    <span style="background:#26241d;color:#7ee7c6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;font-family:monospace;">g = (f ⊕ B) ⊖ B</span>
  </div>

  <div style="padding:16px;background:#ffffff;">
    <p style="font-size:11px;color:#777;margin-bottom:12px;text-align:center;">Fai clic sulle celle di <b>f originale</b> per accendere o spegnere i pixel (riempi i buchi interni!) e regola la dimensione dell'elemento strutturante B.</p>

    <!-- Controle do Tamanho de B -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;margin-bottom:16px;text-align:center;">
      <label style="font-size:11px;font-weight:700;color:#2c7a7b;">Dimensione di B (Scatola n×n)</label><br>
      <input type="range" id="sim_ep0406_sl_n" min="3" max="5" step="2" value="3" style="width:60%;cursor:pointer;accent-color:#2c7a7b;margin-top:6px;">
      <span id="sim_ep0406_vl_n" style="font-family:monospace;font-size:12px;font-weight:700;color:#2c7a7b;margin-left:8px;">3×3</span>
    </div>

    <!-- Pipeline em 3 Colunas: f original vs d (Dilatação) vs g (Fechamento Final) -->
    <div style="display:grid;grid-template-columns:repeat(auto-fit, minmax(170px, 1fr));gap:12px;align-items:start;margin-bottom:14px;">
      
      <!-- f Original -->
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#27ae60;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">f Originale (Cliccabile)</span>
        <div id="sim_ep0406_grid_f" style="display:grid;grid-template-columns:repeat(8, 26px);gap:2px;justify-content:center;user-select:none;"></div>
      </div>

      <!-- d = f ⊕ B -->
      <div style="background:#fafaf7;border:1px solid #2980b9;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#2980b9;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">d = f ⊕ B (Dilatazione)</span>
        <div id="sim_ep0406_grid_d" style="display:grid;grid-template-columns:repeat(8, 26px);gap:2px;justify-content:center;user-select:none;"></div>
      </div>

      <!-- g = d ⊖ B -->
      <div style="background:#fafaf7;border:2px solid #2c7a7b;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#2c7a7b;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">g = d ⊖ B (Chiusura)</span>
        <div id="sim_ep0406_grid_g" style="display:grid;grid-template-columns:repeat(8, 26px);gap:2px;justify-content:center;user-select:none;"></div>
      </div>

    </div>

    <!-- Botões de Ação -->
    <div style="text-align:center;display:flex;gap:8px;justify-content:center;flex-wrap:wrap;">
      <button id="sim_ep0406_btnNew" style="padding:6px 12px;font-size:11px;font-weight:700;border:1px solid #26241d;background:#26241d;color:#7ee7c6;cursor:pointer;border-radius:9px;white-space:nowrap;font-family:monospace;transition:all 0.15s ease;">🎲 Nuova Immagine (Con Buchi)</button>
      <button id="sim_ep0406_btnClear" style="padding:6px 12px;font-size:11px;font-weight:600;border:1px solid #e4dcc8;background:#f1ead7;color:#5e5a4a;cursor:pointer;border-radius:9px;white-space:nowrap;transition:all 0.15s ease;">🧹 Pulisci Tutto</button>
    </div>

  </div>
</div>

<script>
(function(){
  function initSimEP0406(root){
    if (!root || root.dataset.simEp0406Init) return;
    root.dataset.simEp0406Init = "1";

    var slN     = root.querySelector('#sim_ep0406_sl_n');
    var vlN     = root.querySelector('#sim_ep0406_vl_n');
    var gF      = root.querySelector('#sim_ep0406_grid_f');
    var gD      = root.querySelector('#sim_ep0406_grid_d');
    var gG      = root.querySelector('#sim_ep0406_grid_g');
    var btnNew  = root.querySelector('#sim_ep0406_btnNew');
    var btnClear= root.querySelector('#sim_ep0406_btnClear');

    var L = 8, C = 8, f = [];

    function generate() {
      f = [];
      for (var y = 0; y < L; y++) {
        var row = [];
        for (var x = 0; x < C; x++) row.push(0);
        f.push(row);
      }
      for (var y = 1; y < 7; y++) {
        for (var x = 2; x < 6; x++) f[y][x] = 1;
      }
      f[3][3] = 0;
      f[4][4] = 0;
    }

    function clearAll() {
      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) f[y][x] = 0;
      }
    }

    function box(n) {
      var B = [];
      for (var i = 0; i < n; i++) {
        var row = [];
        for (var j = 0; j < n; j++) row.push(1);
        B.push(row);
      }
      return B;
    }

    function reflect(M) {
      var n = M.length, out = [];
      for (var i = n - 1; i >= 0; i--) out.push(M[i].slice().reverse());
      return out;
    }

    function dilate(img, Bm) {
      var Bref = reflect(Bm);
      var HB = Bref.length, WB = Bref[0].length, oy = -HB / 2 + 0.5, ox = -WB / 2 + 0.5;
      var g = [];
      for (var y = 0; y < L; y++) g.push(img[y].slice());

      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) {
          for (var by = 0; by < HB; by++) {
            for (var bx = 0; bx < WB; bx++) {
              if (Bref[by][bx] !== 1) continue;
              var vy = Math.trunc(y + by + oy), vx = Math.trunc(x + bx + ox);
              if (vy >= 0 && vy < L && vx >= 0 && vx < C && img[vy][vx] > g[y][x]) {
                g[y][x] = img[vy][vx];
              }
            }
          }
        }
      }
      return g;
    }

    function erode(img, Bm) {
      var HB = Bm.length, WB = Bm[0].length, oy = -HB / 2 + 0.5, ox = -WB / 2 + 0.5;
      var g = [];
      for (var y = 0; y < L; y++) g.push(img[y].slice());

      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) {
          for (var by = 0; by < HB; by++) {
            for (var bx = 0; bx < WB; bx++) {
              if (Bm[by][bx] !== 1) continue;
              var vy = Math.trunc(y + by + oy), vx = Math.trunc(x + bx + ox);
              if (vy >= 0 && vy < L && vx >= 0 && vx < C && img[vy][vx] < g[y][x]) {
                g[y][x] = img[vy][vx];
              }
            }
          }
        }
      }
      return g;
    }

    function paintStatic(grid, img) {
      grid.innerHTML = '';
      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) {
          var c = document.createElement('div');
          c.style.cssText = 'width:26px;height:26px;border-radius:4px;border:1px solid #e4dcc8;box-sizing:border-box;';
          c.style.background = img[y][x] === 1 ? '#2c7a7b' : '#fafaf7';
          grid.appendChild(c);
        }
      }
    }

    function paintEditable(grid, img) {
      grid.innerHTML = '';
      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) {
          (function(yy, xx){
            var c = document.createElement('div');
            c.style.cssText = 'width:26px;height:26px;border-radius:4px;border:1px solid #e4dcc8;cursor:pointer;box-sizing:border-box;transition:all 0.1s ease;';
            c.style.background = img[yy][xx] === 1 ? '#2c7a7b' : '#fafaf7';

            c.addEventListener('click', function(){
              f[yy][xx] = 1 - f[yy][xx];
              render();
            });
            grid.appendChild(c);
          })(y, x);
        }
      }
    }

    function render() {
      var n = parseInt(slN.value, 10) || 3;
      vlN.textContent = n + '×' + n;
      var B = box(n);
      var d = dilate(f, B);
      var g = erode(d, B);

      paintEditable(gF, f);
      paintStatic(gD, d);
      paintStatic(gG, g);
    }

    slN.addEventListener('input', render);

    btnNew.addEventListener('click', function(){
      generate();
      render();
    });

    btnClear.addEventListener('click', function(){
      clearAll();
      render();
    });

    generate();
    render();
  }

  function tryInitSimEP0406(){
    var root = document.getElementById('sim-ep0406-fechamento');
    if (root) initSimEP0406(root); else setTimeout(tryInitSimEP0406, 200);
  }
  tryInitSimEP0406();
})();
</script>
</div>
""")

**Figura 4.6:** Simulatore EP04_06: Chiusura Morfologica (g = (f ⊕ B) ⊖ B)


<figure id="fig-04-sim-ep0406-fechamento">
  <img src="imagens/fig-04-sim-ep0406-fechamento.png" alt=" Simulatore EP04_06: Chiusura Morfologica (g = (f ⊕ B) ⊖ B) " style="max-width:80%" />
  <figcaption><strong>Figura 4.6:</strong>  Simulatore EP04_06: Chiusura Morfologica (g = (f ⊕ B) ⊖ B) </figcaption>
</figure>

In [ ]:
%%writefile EP04_06.cpp
// your solution

In [ ]:
TestSuite("EP04_06.cpp").run()

### EP04_07 ⛰️ Dilatazione ed Erosione con Pesi (mm.dil1 / mm.ero1)

Finora, l'elemento strutturante diceva solo "questo vicino conta" o "non conta" — ma nei **modelli digitali di elevazione** (usati nei GIS e nella pianificazione del drenaggio urbano), ogni vicino dovrebbe avere un **peso diverso** a seconda della distanza o della direzione del rilievo. Le versioni **ponderate** della dilatazione e dell'erosione, implementate in `morph.py` come `mm::dil1(f, b)` e `mm::ero1(f, b)`, sommano (o sottraggono) il peso di ciascun vicino prima di prendere il massimo (o il minimo) — generalizzando tutto ciò che è stato fatto negli EP precedenti.
Vedere in [Figura 4.7](#fig-04-sim-ep0407-pesos) una simulazione di questo EP.

#### 📋 Linee Guida di Implementazione

1. **Dimensioni dell'immagine:** Leggere gli interi $L$ (righe) e $C$ (colonne) di $f$.
2. **Dimensioni di $b$:** Leggere gli interi $L_B$ (righe) e $C_B$ (colonne) dell'elemento strutturante ponderato.
3. **Pesi:** Leggere la matrice $b$ di pesi **interi** (possono essere negativi, zero o positivi), riga per riga.
4. **Dati:** Leggere la matrice $f$ (l'immagine originale), riga per riga.
5. **Vicinato senza padding:** Per ogni pixel $(y,x)$, percorrere **tutte** le posizioni $(by,bx)$ di $b$ (non solo dove varrebbe $1$ — qui **tutto** il peso partecipa), usando lo stesso spostamento degli EP precedenti:
$$
v_y = y + by + o_y,\quad v_x = x + bx + o_x,\quad o_y=-\tfrac{L_B}{2}+0{,}5,\quad o_x=-\tfrac{C_B}{2}+0{,}5
$$
**Scartare** ogni $(v_y,v_x)$ fuori da $[0,L)\times[0,C)$.
6. **Dilatazione ponderata:** Calcolare
$$
g_{dil}(y,x) = \max\Big(f(y,x),\ \max_{(v_y,v_x)\ \text{valido}} \big(f(v_y,v_x) + b(by,bx)\big)\Big)
$$
7. **Erosione ponderata:** Calcolare, **usando lo stesso $b$ e senza riflessione**:
$$
g_{ero}(y,x) = \min\Big(f(y,x),\ \min_{(v_y,v_x)\ \text{valido}} \big(f(v_y,v_x) - b(by,bx)\big)\Big)
$$
8. **Uscita:** Mostrare **prima** la matrice completa $g_{dil}$, e **poi** la matrice completa $g_{ero}$.

#### 📌 Vincoli Computazionali

* **Nessuna delle due riflette $b$** — la versione ponderata non usa riflessione, nemmeno nella dilatazione (diversamente da `mm::dil0`).
* **Tutti i pesi partecipano:** Non esiste qui il filtro "$B=1$"; anche il peso $0$ entra nel calcolo.
* **Senza padding:** i vicini fuori dall'immagine sono ignorati, mai virtualmente riempiti.
* **Tipo:** L'uscita può contenere valori negativi o maggiori di $255$ — **non** c'è *clipping* in questo EP.
* **Suggerimento:** Per rimuovere i messaggi di overflow quando si superano i limiti del tipo uint8, includere all'inizio del codice:
```python
import warnings
warnings.filterwarnings("ignore")
```

#### 🧠 Fondamenti Teorici

| Concetto | Significato | Impatto Visivo |
|----------|-------------|-----------------|
| **Peso positivo** | "Spinge" il valore del vicino verso l'alto nella dilatazione | Simula un rilievo che sale in quella direzione |
| **Peso negativo** | Riduce il contributo del vicino | Simula distanza o attenuazione direzionale |
| **Dualità ponderata** | $\text{ero1}(f,b) = -\text{dil1}(-f,b)$ | La simmetria tra le due operazioni si mantiene anche con i pesi |

#### 📦 Specifica di Input e Output (VPL)

**Input:**

* Riga 1: Intero $L$.
* Riga 2: Intero $C$.
* Riga 3: Intero $L_B$.
* Riga 4: Intero $C_B$.
* Prossime $L_B$ righe: elementi interi (possono essere negativi) della matrice $b$.
* Prossime $L$ righe: elementi interi della matrice $f$.

**Output:**

* Prima la matrice $g_{dil}$ in $L$ righe e $C$ colonne.
* Successivamente la matrice $g_{ero}$ in $L$ righe e $C$ colonne.

#### 📌 Esempi

| Input | Output | Osservazione |
|---------|-------|------------|
| 3<br>3<br>3<br>3<br>0 1 0<br>1 2 1<br>0 1 0<br>10 20 30<br>40 50 60<br>70 80 90 | 50 60 61<br>80 90 91<br>81 91 92<br>8 9 19<br>9 10 20<br>39 40 50 | Il peso centrale $2$ accelera la crescita nella dilatazione e il restringimento nell'erosione |

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0407-pesos" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

  <!-- Cabeçalho -->
  <div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241d;">⛰️ Simulatore EP04_07: Pesi nell'Elemento Strutturante</span>
    <span style="background:#26241d;color:#7ee7c6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;font-family:monospace;">dil1 / ero1</span>
  </div>

  <div style="padding:16px;background:#ffffff;">
    <p style="font-size:11px;color:#777;margin-bottom:12px;text-align:center;">Regola i pesi dell'elemento strutturante b con i cursori e osserva l'effetto di dilatazione ed erosione con pesi sulla matrice f.</p>

    <!-- Painel dos Pesos b -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:14px;margin-bottom:16px;text-align:center;">
      <span style="font-size:10px;font-weight:700;color:#d35400;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:8px;">Pesi b (Regola i Cursori per Cella)</span>
      <div id="sim_ep0407_grid_b" style="display:grid;grid-template-columns:repeat(3, 70px);gap:8px;justify-content:center;user-select:none;"></div>
    </div>

    <!-- Comparativo em 3 Colunas: f original vs dil1 vs ero1 -->
    <div style="display:grid;grid-template-columns:repeat(auto-fit, minmax(180px, 1fr));gap:14px;align-items:start;margin-bottom:14px;">
      
      <!-- f Original -->
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#27ae60;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">f Originale</span>
        <div id="sim_ep0407_grid_f" style="display:grid;grid-template-columns:repeat(3, 52px);gap:4px;justify-content:center;user-select:none;"></div>
      </div>

      <!-- dil1(f,b) -->
      <div style="background:#fafaf7;border:1px solid #27ae60;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#27ae60;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">dil1(f, b) (Dilatazione)</span>
        <div id="sim_ep0407_grid_d" style="display:grid;grid-template-columns:repeat(3, 52px);gap:4px;justify-content:center;user-select:none;"></div>
      </div>

      <!-- ero1(f,b) -->
      <div style="background:#fafaf7;border:1px solid #c0392b;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#c0392b;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">ero1(f, b) (Erosione)</span>
        <div id="sim_ep0407_grid_e" style="display:grid;grid-template-columns:repeat(3, 52px);gap:4px;justify-content:center;user-select:none;"></div>
      </div>

    </div>

  </div>
</div>

<script>
(function(){
  function initSimEP0407(root){
    if (!root || root.dataset.simEp0407Init) return;
    root.dataset.simEp0407Init = "1";

    var gB = root.querySelector('#sim_ep0407_grid_b');
    var gF = root.querySelector('#sim_ep0407_grid_f');
    var gD = root.querySelector('#sim_ep0407_grid_d');
    var gE = root.querySelector('#sim_ep0407_grid_e');

    var b = [[0, 1, 0], [1, 2, 1], [0, 1, 0]];
    var f = [[10, 20, 30], [40, 50, 60], [70, 80, 90]];
    var L = 3, C = 3;

    function compute() {
      var oy = -3 / 2 + 0.5, ox = -3 / 2 + 0.5;
      var dil = f.map(function(r){ return r.slice(); });
      var ero = f.map(function(r){ return r.slice(); });

      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) {
          for (var by = 0; by < 3; by++) {
            for (var bx = 0; bx < 3; bx++) {
              var vy = Math.trunc(y + by + oy), vx = Math.trunc(x + bx + ox);
              if (vy >= 0 && vy < L && vx >= 0 && vx < C) {
                var cd = f[vy][vx] + b[by][bx];
                if (cd > dil[y][x]) dil[y][x] = cd;
                var ce = f[vy][vx] - b[by][bx];
                if (ce < ero[y][x]) ero[y][x] = ce;
              }
            }
          }
        }
      }
      return { dil: dil, ero: ero };
    }

    function renderB() {
      gB.innerHTML = '';
      for (var by = 0; by < 3; by++) {
        for (var bx = 0; bx < 3; bx++) {
          (function(row, col){
            var wrap = document.createElement('div');
            wrap.style.cssText = 'display:flex;flex-direction:column;align-items:center;background:#fafaf7;border:1px solid #e4dcc8;border-radius:6px;padding:4px;box-sizing:border-box;';

            var val = document.createElement('div');
            val.style.cssText = 'font-family:monospace;font-weight:700;font-size:11px;color:#d35400;margin-bottom:2px;';
            val.textContent = b[row][col];

            var sl = document.createElement('input');
            sl.type = 'range';
            sl.min = '-5';
            sl.max = '5';
            sl.step = '1';
            sl.value = b[row][col];
            sl.style.cssText = 'width:56px;cursor:pointer;accent-color:#d35400;';

            sl.addEventListener('input', function(){
              b[row][col] = parseInt(sl.value, 10);
              val.textContent = b[row][col];
              renderAll();
            });

            wrap.appendChild(val);
            wrap.appendChild(sl);
            gB.appendChild(wrap);
          })(by, bx);
        }
      }
    }

    function paint(grid, img) {
      grid.innerHTML = '';
      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) {
          var c = document.createElement('div');
          var v = img[y][x];
          var inten = Math.min(255, Math.max(0, v));
          var fg = inten > 128 ? '#000000' : '#ffffff';
          c.style.cssText = 'width:52px;height:42px;display:flex;align-items:center;justify-content:center;font-size:11px;font-weight:700;font-family:monospace;border-radius:6px;border:1px solid #e4dcc8;user-select:none;background:rgb(' + inten + ',' + inten + ',' + inten + ');color:' + fg + ';box-sizing:border-box;';
          c.textContent = v;
          grid.appendChild(c);
        }
      }
    }

    function renderAll() {
      var res = compute();
      paint(gF, f);
      paint(gD, res.dil);
      paint(gE, res.ero);
    }

    renderB();
    renderAll();
  }

  function tryInitSimEP0407(){
    var root = document.getElementById('sim-ep0407-pesos');
    if (root) initSimEP0407(root); else setTimeout(tryInitSimEP0407, 200);
  }
  tryInitSimEP0407();
})();
</script>
</div>
""")

**Figura 4.7:** Simulatore EP04_07: Dilatazione ed Erosione con Pesi (mm.dil1 / mm.ero1)


<figure id="fig-04-sim-ep0407-pesos">
  <img src="imagens/fig-04-sim-ep0407-pesos.png" alt=" Simulatore EP04_07: Dilatazione ed Erosione con Pesi (mm.dil1 / mm.ero1) " style="max-width:80%" />
  <figcaption><strong>Figura 4.7:</strong>  Simulatore EP04_07: Dilatazione ed Erosione con Pesi (mm.dil1 / mm.ero1) </figcaption>
</figure>

In [ ]:
%%writefile EP04_07.cpp
// your solution

In [ ]:
TestSuite("EP04_07.cpp").run()

### EP04_08 🌋 Gradiente Morfologico, Top-hat e Black-hat

Nell'**ispezione automatica dei circuiti stampati**, tre domande ricorrono continuamente: dove sono i **bordi** dei componenti? Quali **dettagli chiari e piccoli** (come i punti di saldatura) si distinguono dallo sfondo? Quali **incavi scuri** (come le cricche) lo sfondo nasconde? Una singola coppia erosione/dilatazione risponde a tutte e tre: il **gradiente morfologico** evidenzia i contorni, il **top-hat** rivela i picchi stretti, e il **black-hat** rivela le valli strette — tre strumenti, un solo intorno.
Vedi in [Figura 4.8](#fig-04-sim-ep0408-gradiente) una simulazione di questo EP.

#### 📋 Linee Guida di Implementazione

1. **Dimensioni dell'immagine:** Leggere gli interi $L$ (righe) e $C$ (colonne) di $f$.
2. **Dimensioni di $B$:** Leggere gli interi $L_B$ (righe) e $C_B$ (colonne) dell'elemento strutturante.
3. **Elemento strutturante:** Leggere la matrice $B$ con valori $0$ o $1$, riga per riga.
4. **Dati:** Leggere la matrice $f$ (l'immagine originale, in scala di grigi), riga per riga.
5. **Operatori di base:** Calcolare, esattamente come negli EP 04_03 fino a 04_06:
   * $d = f \oplus B$ (dilatazione),
   * $e = f \ominus B$ (erosione),
   * $\text{ apertura} = e \oplus B$,
   * $\text{chiusura} = d \ominus B$.
6. **Gradiente morfologico:** $\text{grad}(y,x) = d(y,x) - e(y,x)$.
7. **Top-hat:** $\text{tophat}(y,x) = f(y,x) - \text{apertura}(y,x)$.
8. **Black-hat:** $\text{blackhat}(y,x) = \text{chiusura}(y,x) - f(y,x)$.
9. **Output:** Mostrare, **in questo ordine**, le tre matrici complete: gradiente, top-hat, black-hat.

#### 📌 Vincoli Computazionali

* **Nessun padding in nessuna fase intermedia** — dilatazione, erosione, apertura e chiusura seguono le stesse regole di vicinato degli EP precedenti.
* **Nessun *clipping*:** le tre uscite possono contenere qualsiasi valore intero (il gradiente è sempre $\geq 0$, ma anche top-hat e black-hat lo sono).
* **Riutilizzo:** $d$ e $e$ devono essere calcolati **una sola volta** e riutilizzati per costruire apertura, chiusura e gradiente.

#### 🧠 Fondamenti Teorici

| Operatore | Formula | Cosa rivela |
|-----------|---------|-------------|
| **Gradiente** | $d - e$ | Bordi: zero in regioni piatte, alto nelle transizioni |
| **Top-hat** | $f - \text{apertura}(f)$ | Elementi **chiari e sottili**, più piccoli di $B$ |
| **Black-hat** | $\text{chiusura}(f) - f$ | Elementi **scuri e sottili**, più piccoli di $B$ |

#### 📦 Specifica di Input e Output (VPL)

**Input:**

* Riga 1: Intero $L$.
* Riga 2: Intero $C$.
* Riga 3: Intero $L_B$.
* Riga 4: Intero $C_B$.
* Prossime $L_B$ righe: elementi interi ($0$ o $1$) della matrice $B$.
* Prossime $L$ righe: elementi interi della matrice $f$.

**Output:**

* Matrice gradiente in $L$ righe e $C$ colonne.
* Matrice top-hat in $L$ righe e $C$ colonne.
* Matrice black-hat in $L$ righe e $C$ colonne.

#### 📌 Esempi

| Input | Output | Osservazione |
|-------|--------|--------------|
| 9<br>9<br>3<br>3<br>1 1 1<br>1 1 1<br>1 1 1<br>10 10 10 10 10 10 10 10 10<br>10 10 10 10 10 10 10 10 10<br>10 10 80 10 10 10 10 10 10<br>10 10 10 10 10 10 10 10 10<br>10 10 10 10 10 10 10 10 10<br>10 10 10 10 10 10 10 10 10<br>10 10 10 10 10 10 2 10 10<br>10 10 10 10 10 10 10 10 10<br>10 10 10 10 10 10 10 10 10 | (gradiente: alone $3\times3=70$ attorno a $(2,2)$ e alone $3\times3=8$ attorno a $(6,6)$, resto $0$)<br>(top-hat: unico $70$ in $(2,2)$, resto $0$)<br>(black-hat: unico $8$ in $(6,6)$, resto $0$) | Picco isolato diventa top-hat; valle isolata diventa black-hat; entrambi appaiono nel gradiente |

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0408-gradiente" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

  <!-- Cabeçalho -->
  <div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241d;">🌋 Simulatore EP04_08: Gradiente / Top-hat / Black-hat</span>
    <span style="background:#26241d;color:#7ee7c6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;font-family:monospace;">3 operatori, 1 vicinanza</span>
  </div>

  <div style="padding:16px;background:#ffffff;">
    <p style="font-size:11px;color:#777;margin-bottom:12px;text-align:center;">Aggiungi picchi o valli nella matrice f e osserva il comportamento simultaneo degli operatori gradiente, top-hat e black-hat.</p>

    <!-- Botões de Ação -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;margin-bottom:16px;text-align:center;display:flex;gap:8px;justify-content:center;flex-wrap:wrap;">
      <button id="sim_ep0408_btn_pico" style="padding:6px 12px;font-size:11px;font-weight:700;border:1px solid #b9770e;background:#fef5e7;color:#b9770e;cursor:pointer;border-radius:8px;transition:all 0.15s ease;">☀️ Aggiungi Picco</button>
      <button id="sim_ep0408_btn_vale" style="padding:6px 12px;font-size:11px;font-weight:700;border:1px solid #2980b9;background:#ebf4fd;color:#2980b9;cursor:pointer;border-radius:8px;transition:all 0.15s ease;">🕳️ Aggiungi Valle</button>
      <button id="sim_ep0408_btn_reset" style="padding:6px 12px;font-size:11px;font-weight:600;border:1px solid #e4dcc8;background:#f1ead7;color:#5e5a4a;cursor:pointer;border-radius:8px;transition:all 0.15s ease;">↩ Cancella Tutto</button>
    </div>

    <!-- Comparativo em 4 Colunas: f, Gradiente, Top-hat, Black-hat -->
    <div style="display:grid;grid-template-columns:repeat(auto-fit, minmax(140px, 1fr));gap:12px;align-items:start;margin-bottom:14px;">
      
      <!-- Matriz f -->
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:10px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#7f8c8d;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:8px;">f (Ingresso)</span>
        <div id="sim_ep0408_grid_f" style="display:grid;grid-template-columns:repeat(9, 20px);gap:1px;justify-content:center;user-select:none;"></div>
      </div>

      <!-- Gradiente -->
      <div style="background:#fafaf7;border:1px solid #8e44ad;border-radius:12px;padding:10px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#8e44ad;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:8px;">Gradiente</span>
        <div id="sim_ep0408_grid_grad" style="display:grid;grid-template-columns:repeat(9, 20px);gap:1px;justify-content:center;user-select:none;"></div>
      </div>

      <!-- Top-hat -->
      <div style="background:#fafaf7;border:1px solid #d35400;border-radius:12px;padding:10px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#d35400;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:8px;">Top-hat</span>
        <div id="sim_ep0408_grid_th" style="display:grid;grid-template-columns:repeat(9, 20px);gap:1px;justify-content:center;user-select:none;"></div>
      </div>

      <!-- Black-hat -->
      <div style="background:#fafaf7;border:1px solid #2980b9;border-radius:12px;padding:10px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#2980b9;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:8px;">Black-hat</span>
        <div id="sim_ep0408_grid_bh" style="display:grid;grid-template-columns:repeat(9, 20px);gap:1px;justify-content:center;user-select:none;"></div>
      </div>

    </div>

  </div>
</div>

<script>
(function(){
  function initSimEP0408(root){
    if (!root || root.dataset.simEp0408Init) return;
    root.dataset.simEp0408Init = "1";

    var gF   = root.querySelector('#sim_ep0408_grid_f');
    var gGrad= root.querySelector('#sim_ep0408_grid_grad');
    var gTh  = root.querySelector('#sim_ep0408_grid_th');
    var gBh  = root.querySelector('#sim_ep0408_grid_bh');

    var btnPico  = root.querySelector('#sim_ep0408_btn_pico');
    var btnVale  = root.querySelector('#sim_ep0408_btn_vale');
    var btnReset = root.querySelector('#sim_ep0408_btn_reset');

    var L = 9, C = 9, f = [], B = [[1, 1, 1], [1, 1, 1], [1, 1, 1]];

    function resetMatrix() {
      f = Array.from({ length: L }, function(){ return new Array(C).fill(10); });
    }

    function morph(img, Bm, mode) {
      var Bref = mode === 'dil' ? Bm.slice().reverse().map(function(r){ return r.slice().reverse(); }) : Bm;
      var HB = Bref.length, WB = Bref[0].length, oy = -HB / 2 + 0.5, ox = -WB / 2 + 0.5;
      var g = img.map(function(r){ return r.slice(); });

      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) {
          for (var by = 0; by < HB; by++) {
            for (var bx = 0; bx < WB; bx++) {
              if (Bref[by][bx] !== 1) continue;
              var vy = Math.trunc(y + by + oy), vx = Math.trunc(x + bx + ox);
              if (vy >= 0 && vy < L && vx >= 0 && vx < C) {
                if (mode === 'dil' && img[vy][vx] > g[y][x]) g[y][x] = img[vy][vx];
                if (mode === 'ero' && img[vy][vx] < g[y][x]) g[y][x] = img[vy][vx];
              }
            }
          }
        }
      }
      return g;
    }

    function paint(grid, img, cmin, cmax, hue) {
      grid.innerHTML = '';
      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) {
          var v = img[y][x];
          var t = cmax > cmin ? (v - cmin) / (cmax - cmin) : 0;
          var c = document.createElement('div');
          c.style.cssText = 'width:20px;height:20px;border-radius:3px;box-sizing:border-box;';
          c.style.background = v === 0 ? '#fafaf7' : hue;
          c.style.opacity = v === 0 ? '1' : (0.35 + 0.65 * Math.min(1, t));
          grid.appendChild(c);
        }
      }
    }

    function render() {
      var d = morph(f, B, 'dil');
      var e = morph(f, B, 'ero');
      var ab = morph(e, B, 'dil');
      var fc = morph(d, B, 'ero');

      var grad = f.map(function(r, y){ return r.map(function(_, x){ return d[y][x] - e[y][x]; }); });
      var th   = f.map(function(r, y){ return r.map(function(v, x){ return v - ab[y][x]; }); });
      var bh   = f.map(function(r, y){ return r.map(function(v, x){ return fc[y][x] - v; }); });

      var maxF = Math.max.apply(null, f.map(function(r){ return Math.max.apply(null, r); }));
      var maxG = Math.max.apply(null, grad.map(function(r){ return Math.max.apply(null, r); }));
      var maxTh = Math.max.apply(null, th.map(function(r){ return Math.max.apply(null, r); }));
      var maxBh = Math.max.apply(null, bh.map(function(r){ return Math.max.apply(null, r); }));

      paint(gF, f, 10, maxF || 1, '#7f8c8d');
      paint(gGrad, grad, 0, Math.max(1, maxG), '#8e44ad');
      paint(gTh, th, 0, Math.max(1, maxTh), '#d35400');
      paint(gBh, bh, 0, Math.max(1, maxBh), '#2980b9');
    }

    btnPico.addEventListener('click', function(){
      var y = 2 + Math.floor(Math.random() * 5), x = 2 + Math.floor(Math.random() * 5);
      f[y][x] = Math.min(255, f[y][x] + 60 + Math.floor(Math.random() * 30));
      render();
    });

    btnVale.addEventListener('click', function(){
      var y = 2 + Math.floor(Math.random() * 5), x = 2 + Math.floor(Math.random() * 5);
      f[y][x] = Math.max(0, f[y][x] - 8 - Math.floor(Math.random() * 4));
      render();
    });

    btnReset.addEventListener('click', function(){
      resetMatrix();
      f[2][2] = 80;
      f[6][6] = 2;
      render();
    });

    resetMatrix();
    f[2][2] = 80;
    f[6][6] = 2;
    render();
  }

  function tryInitSimEP0408(){
    var root = document.getElementById('sim-ep0408-gradiente');
    if (root) initSimEP0408(root); else setTimeout(tryInitSimEP0408, 200);
  }
  tryInitSimEP0408();
})();
</script>
</div>
""")

**Figura 4.8:** Simulatore EP04_08: Gradiente morfologico, Top-hat e Black-hat


<figure id="fig-04-sim-ep0408-gradiente">
  <img src="imagens/fig-04-sim-ep0408-gradiente.png" alt=" Simulatore EP04_08: Gradiente morfologico, Top-hat e Black-hat " style="max-width:80%" />
  <figcaption><strong>Figura 4.8:</strong>  Simulatore EP04_08: Gradiente morfologico, Top-hat e Black-hat </figcaption>
</figure>

In [ ]:
%%writefile EP04_08.cpp
// your solution

In [ ]:
TestSuite("EP04_08.cpp").run()

### EP04_09 🗺️ Trasformata della Distanza e il "Nucleo" dell'Oggetto

Nella **robotica mobile**, quando si pianifica un percorso all'interno di un corridoio, il robot vuole sapere non solo *dove* c'è spazio libero, ma anche **quanto distante** ogni punto libero sia dalla parete più vicina. I percorsi più sicuri tendono a passare attraverso il "nucleo" del corridoio, lontano dagli ostacoli.

La **trasformata della distanza morfologica** assegna a ogni pixel un valore che rappresenta la sua distanza dal bordo più vicino, secondo la metrica definita dall'elemento strutturante. I pixel vicini al bordo ricevono valori bassi, mentre i pixel più interni ricevono valori più alti. Il pixel con il valore massimo corrisponde alla regione più protetta dell'oggetto, spesso associata al suo centro morfologico.

Vedere in [Figura 4.9](#fig-04-sim-ep0409-distancia) una simulazione di questo EP.

#### 📋 Linee Guida di Implementazione

1. **Dimensioni dell'immagine:** leggere gli interi $L$ (righe) e $C$ (colonne) dell'immagine $f$.
2. **Dimensioni di $B$:** leggere gli interi $L_B$ (righe) e $C_B$ (colonne) dell'elemento strutturante.
3. **Elemento strutturante:** leggere la matrice $b$, contenente valore $0$ al centro e valori negativi nelle altre posizioni.
4. **Immagine:** leggere la matrice binaria $f$ (valori $0$ o $1$), riga per riga.
5. **Preparazione:** moltiplicare l'immagine per $L\times C$, garantendo che i pixel interni abbiano un valore iniziale sufficientemente alto per la propagazione delle distanze.
6. **Trasformata della distanza:** calcolare la matrice delle distanze utilizzando il metodo `mm::dist1(f,b)`.
7. **Output:** visualizzare la matrice risultante dalla trasformata della distanza.

#### 📌 Vincoli Computazionali

* Utilizzare l'implementazione dell'erosione ponderata fornita dalla libreria.
* L'elemento strutturante può contenere valori negativi arbitrari.
* La trasformata deve essere ottenuta applicando iterativamente erosioni ponderate fino a raggiungere un punto fisso.

**⚠️ Nota Cruciale sulla Lettura delle Matrici:** Poiché l'elemento strutturante può contenere valori interi negativi (ad esempio, `-1` e `-99`), **non utilizzare la funzione `mm::readImg` per leggere la matrice $b$**. Questa funzione converte i dati nel tipo `uint8`, causando *underflow* e corrompendo i valori negativi. Leggere le $L_B$ righe di $b$ manualmente utilizzando il tipo predefinito `int`. L'immagine $f$ può continuare a essere letta normalmente con `mm::readImg`.

#### 🧠 Fondamenti Teorici

| Concetto                            | Significato                                                                       | Impatto Visivo                               |
| ----------------------------------- | --------------------------------------------------------------------------------- | -------------------------------------------- |
| **$\text{dist}(y,x)$**              | Distanza morfologica fino al bordo più vicino secondo la metrica definita da $b$ | I pixel più interni ricevono valori più alti |
| **Valore massimo**                  | Pixel più distante dal bordo                                                      | Approssima il centro morfologico dell'oggetto|
| **Elemento strutturante ponderato** | Definisce i costi di spostamento tra pixel vicini                                 | Determina la metrica della distanza utilizzata |
| **Oggetti sottili**                 | Regioni strette dell'oggetto                                                      | Producono valori bassi di distanza           |

#### 📦 Specifica di Input e Output (VPL)

**Input:**

* Riga 1: intero $L$.
* Riga 2: intero $C$.
* Riga 3: intero $L_B$.
* Riga 4: intero $C_B$.
* Prossime $L_B$ righe: elementi interi della matrice $b$.
* Prossime $L$ righe: elementi binari ($0$ o $1$) della matrice $f$.

⚠️ **Nota di implementazione:** Gli elementi della matrice $f$ (0 o 1) devono essere moltiplicati per **255** per generare un'immagine binaria adeguata ($0$ e $255$) prima di applicare la Trasformata della Distanza (TD).

**Output:**

* Matrice della trasformata della distanza in $L$ righe e $C$ colonne.

#### 📌 Esempio

| Input                                                                                                                                                            | Output                                                                                                 | Osservazione                             |
| ---------------------------------------------------------------------------------------------------------------------------------------------------------------- | ----------------------------------------------------------------------------------------------------- | ---------------------------------------- |
| 5<br>9<br>3<br>3<br>-99 -1 -99<br>-1 0 -1<br>-99 -1 -99<br>0 0 0 0 0 0 0 0 0<br>0 1 1 1 1 1 1 1 0<br>0 1 1 1 1 1 1 1 0<br>0 1 1 1 1 1 1 1 0<br>0 0 0 0 0 0 0 0 0 | 0 0 0 0 0 0 0 0 0<br>0 1 1 1 1 1 1 1 0<br>0 1 2 2 2 2 2 1 0<br>0 1 1 1 1 1 1 1 0<br>0 0 0 0 0 0 0 0 0 | Risultato della trasformata della distanza. |

**Nota:** il valore `-99` agisce come un'approssimazione pratica di $-\infty$, impedendo la propagazione attraverso le diagonali. In questo modo, solo i vicini orizzontali e verticali contribuiscono alla distanza, producendo la distanza di Manhattan.

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0409-distancia" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

  <!-- Cabeçalho -->
  <div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241d;">🗺️ Simulatore EP04_09: Trasformata della Distanza</span>
    <span style="background:#26241d;color:#7ee7c6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;font-family:monospace;">Livelli di Erosione</span>
  </div>

  <div style="padding:16px;background:#ffffff;">
    <p style="font-size:11px;color:#777;margin-bottom:12px;text-align:center;">Fai clic sulle celle per disegnare il tuo oggetto oppure seleziona una forma predefinita per calcolare la mappa delle distanze a cascata.</p>

    <!-- Grade f Original Clicável -->
    <div style="display:flex;justify-content:center;margin-bottom:14px;">
      <div id="sim_ep0409_grid_f" style="display:grid;grid-template-columns:repeat(9, 32px);gap:2px;user-select:none;"></div>
    </div>

    <!-- Botões de Formas Predefinidas -->
    <div style="text-align:center;margin-bottom:14px;display:flex;gap:8px;justify-content:center;flex-wrap:wrap;">
      <button id="sim_ep0409_btn_corredor" style="padding:6px 12px;font-size:11px;font-weight:700;border:1px solid #16a085;background:#eafaf1;color:#16a085;cursor:pointer;border-radius:8px;transition:all 0.15s ease;">📐 Corridoio</button>
      <button id="sim_ep0409_btn_disco" style="padding:6px 12px;font-size:11px;font-weight:700;border:1px solid #16a085;background:#eafaf1;color:#16a085;cursor:pointer;border-radius:8px;transition:all 0.15s ease;">⬤ Disco</button>
      <button id="sim_ep0409_btn_l" style="padding:6px 12px;font-size:11px;font-weight:700;border:1px solid #16a085;background:#eafaf1;color:#16a085;cursor:pointer;border-radius:8px;transition:all 0.15s ease;">📏 Forma a L</button>
    </div>

    <!-- Título do Mapa de Distâncias -->
    <span style="font-size:10px;font-weight:700;color:#5e5a4a;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:8px;text-align:center;">Mappa delle Distanze Calcolata</span>

    <!-- Grade de Distâncias -->
    <div style="display:flex;justify-content:center;">
      <div id="sim_ep0409_grid_dist" style="display:grid;grid-template-columns:repeat(9, 32px);gap:2px;user-select:none;"></div>
    </div>

  </div>
</div>

<script>
(function(){
  function initSimEP0409(root){
    if (!root || root.dataset.simEp0409Init) return;
    root.dataset.simEp0409Init = "1";

    var gF = root.querySelector('#sim_ep0409_grid_f');
    var gD = root.querySelector('#sim_ep0409_grid_dist');

    var L = 5, C = 9, f = [];
    var B = [[0, 1, 0], [1, 1, 1], [0, 1, 0]];

    function setCorredor() {
      f = Array.from({length: L}, function(){ return new Array(C).fill(0); });
      for (var y = 1; y < 4; y++) {
        for (var x = 1; x < 8; x++) f[y][x] = 1;
      }
    }

    function setDisco() {
      f = Array.from({length: L}, function(){ return new Array(C).fill(0); });
      var cy = 2, cx = 4;
      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) {
          if (Math.pow(y - cy, 2) + Math.pow((x - cx) * 0.6, 2) <= 4) f[y][x] = 1;
        }
      }
    }

    function setL() {
      f = Array.from({length: L}, function(){ return new Array(C).fill(0); });
      for (var y = 1; y < 4; y++) {
        for (var x = 1; x < 3; x++) f[y][x] = 1;
      }
      for (var y = 2; y < 4; y++) {
        for (var x = 1; x < 8; x++) f[y][x] = 1;
      }
    }

    function erode(img, Bm) {
      var HB = Bm.length, WB = Bm[0].length, oy = -HB / 2 + 0.5, ox = -WB / 2 + 0.5;
      var g = img.map(function(r){ return r.slice(); });

      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) {
          for (var by = 0; by < HB; by++) {
            for (var bx = 0; bx < WB; bx++) {
              if (Bm[by][bx] !== 1) continue;
              var vy = Math.trunc(y + by + oy), vx = Math.trunc(x + bx + ox);
              if (vy >= 0 && vy < L && vx >= 0 && vx < C && img[vy][vx] < g[y][x]) {
                g[y][x] = img[vy][vx];
              }
            }
          }
        }
      }
      return g;
    }

    function sameMatrix(a, b) {
      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) {
          if (a[y][x] !== b[y][x]) return false;
        }
      }
      return true;
    }

    function render() {
      gF.innerHTML = '';
      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) {
          (function(yy, xx){
            var c = document.createElement('div');
            c.style.cssText = 'width:32px;height:32px;border-radius:6px;border:1px solid #e4dcc8;cursor:pointer;box-sizing:border-box;transition:all 0.1s ease;';
            c.style.background = f[yy][xx] ? '#16a085' : '#fafaf7';

            c.addEventListener('click', function(){
              f[yy][xx] = 1 - f[yy][xx];
              render();
            });
            gF.appendChild(c);
          })(y, x);
        }
      }

      var dist = Array.from({length: L}, function(){ return new Array(C).fill(0); });
      var atual = f.map(function(r){ return r.slice(); });
      var nivel = 0;

      while (atual.some(function(r){ return r.some(function(v){ return v === 1; }); })) {
        nivel++;
        for (var y = 0; y < L; y++) {
          for (var x = 0; x < C; x++) {
            if (atual[y][x] === 1) dist[y][x] = nivel;
          }
        }
        var prox = erode(atual, B);
        if (sameMatrix(prox, atual)) break;
        atual = prox;
      }

      gD.innerHTML = '';
      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) {
          var c = document.createElement('div');
          var v = dist[y][x];
          var t = v / (nivel || 1);
          var inten = Math.round(220 - t * 170);

          c.style.cssText = 'width:32px;height:32px;border-radius:6px;display:flex;align-items:center;justify-content:center;font-size:11px;font-weight:700;font-family:monospace;border:1px solid #e4dcc8;box-sizing:border-box;';
          c.style.background = v === 0 ? '#fafaf7' : 'rgb(' + (inten - 60) + ',' + inten + ',' + (inten - 30) + ')';
          c.style.color = v > 0 ? '#ffffff' : '#8a8371';
          c.textContent = v || '';
          gD.appendChild(c);
        }
      }
    }

    root.querySelector('#sim_ep0409_btn_corredor').addEventListener('click', function(){ setCorredor(); render(); });
    root.querySelector('#sim_ep0409_btn_disco').addEventListener('click', function(){ setDisco(); render(); });
    root.querySelector('#sim_ep0409_btn_l').addEventListener('click', function(){ setL(); render(); });

    setCorredor();
    render();
  }

  function tryInitSimEP0409(){
    var root = document.getElementById('sim-ep0409-distancia');
    if (root) initSimEP0409(root); else setTimeout(tryInitSimEP0409, 200);
  }
  tryInitSimEP0409();
})();
</script>
</div>
""")

**Figura 4.9:** Simulatore EP04_09: Trasformata della Distanza (Livelli di Erosione)


<figure id="fig-04-sim-ep0409-distancia">
  <img src="imagens/fig-04-sim-ep0409-distancia.png" alt=" Simulatore EP04_09: Trasformata della Distanza (Livelli di Erosione) " style="max-width:80%" />
  <figcaption><strong>Figura 4.9:</strong>  Simulatore EP04_09: Trasformata della Distanza (Livelli di Erosione) </figcaption>
</figure>

In [ ]:
%%writefile EP04_09.cpp
// your solution

In [ ]:
TestSuite("EP04_09.cpp").run()

### EP04_10 🪙 Separazione dei *Blob*, Etichettatura e Descrittori

In una **linea di produzione di monete**, è comune che i pezzi si tocchino l'un l'altro sul nastro trasportatore, formando un'unica macchia connessa nell'immagine — un conteggio ingenuo sbaglierebbe il totale. La soluzione classica combina operazioni morfologiche e analisi di connettività: prima un'**erosione** riduce o spezza le connessioni fragili tra gli oggetti, e poi l'**etichettatura delle componenti connesse** separa ciascun oggetto in una regione distinta. Infine, **descrittori geometrici** (area e bounding box) riassumono ciascuna componente trovata.

Vedi in [Figura 4.10](#fig-04-sim-ep0410-rotulacao) una simulazione di questo EP.

#### 📋 Linee Guida di Implementazione

1. **Dimensioni dell'immagine:** leggere gli interi $L$ (righe) e $C$ (colonne) da $f$.

2. **Dimensioni di $B$:** leggere gli interi $L_B$ (righe) e $C_B$ (colonne) dell'elemento strutturante.

3. **Elemento strutturante:** leggere la matrice $B$, contenente valori $0$ o $1$, riga per riga.

4. **Dati:** leggere la matrice binaria $f$ (valori $0$ o $1$), riga per riga.

5. **Separazione:** calcolare
   $$
   f_{ero} = f \ominus B
   $$
   usando erosione binaria piatta (come nell'EP04_04), eliminando connessioni fragili tra gli oggetti.

6. **Etichettatura:** su $f_{ero}$, identificare le componenti connesse usando la connettività definita dall'intorno $B$. L'etichettatura deve seguire una scansione *raster*: quando si trova un pixel $1$ non ancora etichettato, assegnare una nuova etichetta intera crescente a partire da 1 e propagare tale etichetta a tutta la regione connessa.

7. **Descrittori:** per ogni etichetta $k$, calcolare:

   * **Area:** numero di pixel appartenenti all'etichetta;
   * **Bounding box:** $$(y_{min}, x_{min}, y_{max}, x_{max})$$

8. **Output:** mostrare il numero totale di etichette e, successivamente, una riga per etichetta nel formato:
   $$
   k,\ \text{area},\ y_{min},\ x_{min},\ y_{max},\ x_{max}
   $$

#### 📌 Vincoli Computazionali

* L'erosione deve essere applicata prima dell'etichettatura.
* La connettività è fissa e definita dall'intorno sopra descritto.
* L'elemento strutturante $B$ non interferisce con la connettività dell'etichettatura.
* Nessun padding in alcuna fase.
* L'ordine delle etichette segue la prima scoperta durante la scansione *raster*.

#### 🧠 Fondamenti Teorici

| Concetto          | Significato                                  | Impatto                                              |
| ----------------- | -------------------------------------------- | ---------------------------------------------------- |
| Ponte sottile     | Connessione stretta tra oggetti              | Può essere rimosso dall'erosione morfologica         |
| Connettività      | Definita dall'insieme $$\mathcal{N}(y,x)$$   | Determina quali pixel appartengono alla stessa componente |
| Area              | Numero di pixel per componente               | Stima diretta della dimensione dell'oggetto           |
| Bounding box      | Estensione spaziale dell'etichetta           | Riassunto geometrico della componente                 |

#### 📦 Specifica di Input e Output (VPL)

**Input:**

* Riga 1: intero $L$
* Riga 2: intero $C$
* Riga 3: intero $L_B$
* Riga 4: intero $C_B$
* Prossime $L_B$ righe: matrice $B$
* Prossime $L$ righe: matrice $f$

**Output:**

* Riga 1: numero totale di etichette trovate
* Righe successive:
  $$
  k,\ \text{area},\ y_{min},\ x_{min},\ y_{max},\ x_{max}
  $$

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0410-rotulacao" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

  <!-- Cabeçalho -->
  <div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241d;">🪙 Simulatore EP04_10: Monete Attaccate → Separate → Contate</span>
    <span style="background:#26241d;color:#7ee7c6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;font-family:monospace;">erosione + etichetta + descrittori</span>
  </div>

  <div style="padding:16px;background:#ffffff;">
    <p style="font-size:11px;color:#777;margin-bottom:12px;text-align:center;">Regola lo spessore del ponte tra le monete e osserva come l'erosione morfologica separa gli oggetti per il conteggio e l'estrazione dei descrittori (area e bounding box).</p>

    <!-- Controle de Espessura da Ponte -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;margin-bottom:16px;text-align:center;">
      <label style="font-size:11px;font-weight:700;color:#b9770e;">Spessore del Ponte tra le Monete</label><br>
      <input type="range" id="sim_ep0410_sl_p" min="1" max="3" step="1" value="1" style="width:60%;cursor:pointer;accent-color:#b9770e;margin-top:6px;">
      <span id="sim_ep0410_vl_p" style="font-family:monospace;font-size:12px;font-weight:700;color:#b9770e;margin-left:8px;">1 px</span>
    </div>

    <!-- Comparativo Lado a Lado: f original vs Rótulos Pós-Erosão -->
    <div style="display:grid;grid-template-columns:repeat(auto-fit, minmax(200px, 1fr));gap:16px;align-items:start;margin-bottom:14px;">
      
      <!-- f Original (ligadas) -->
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#27ae60;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">f Originale (Collegate)</span>
        <div id="sim_ep0410_grid_f" style="display:grid;grid-template-columns:repeat(10, 26px);gap:2px;justify-content:center;user-select:none;"></div>
      </div>

      <!-- Após Erosão + Rótulos -->
      <div style="background:#fafaf7;border:1px solid #2980b9;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#2980b9;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">Dopo Erosione + Etichette</span>
        <div id="sim_ep0410_grid_lab" style="display:grid;grid-template-columns:repeat(10, 26px);gap:2px;justify-content:center;user-select:none;"></div>
      </div>

    </div>

    <!-- Painel Informativo / Descritores -->
    <div id="sim_ep0410_info" style="background:#fef5e7;border:1px solid #f8c471;border-radius:8px;padding:10px 14px;font-size:11px;color:#7d5a00;text-align:center;line-height:1.5;"></div>

  </div>
</div>

<script>
(function(){
  function initSimEP0410(root){
    if (!root || root.dataset.simEp0410Init) return;
    root.dataset.simEp0410Init = "1";

    var slP  = root.querySelector('#sim_ep0410_sl_p');
    var vlP  = root.querySelector('#sim_ep0410_vl_p');
    var gF   = root.querySelector('#sim_ep0410_grid_f');
    var gL   = root.querySelector('#sim_ep0410_grid_lab');
    var info = root.querySelector('#sim_ep0410_info');

    var L = 7, C = 10;
    var B = [[1, 1, 1], [1, 1, 1], [1, 1, 1]];
    var palette = ['#e74c3c', '#27ae60', '#2980b9', '#8e44ad', '#d35400'];

    function buildF(p) {
      var f = Array.from({length: L}, function(){ return new Array(C).fill(0); });
      for (var y = 1; y < 6; y++) {
        for (var x = 1; x < 4; x++) f[y][x] = 1;
      }
      for (var y = 1; y < 6; y++) {
        for (var x = 6; x < 9; x++) f[y][x] = 1;
      }
      var midRow = 3;
      for (var dy = 0; dy < p; dy++) {
        var ry = midRow - Math.floor(p / 2) + dy;
        for (var x = 4; x < 6; x++) f[ry][x] = 1;
      }
      return f;
    }

    function erode(img, Bm) {
      var HB = Bm.length, WB = Bm[0].length, oy = -HB / 2 + 0.5, ox = -WB / 2 + 0.5;
      var g = img.map(function(r){ return r.slice(); });

      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) {
          for (var by = 0; by < HB; by++) {
            for (var bx = 0; bx < WB; bx++) {
              if (Bm[by][bx] !== 1) continue;
              var vy = Math.trunc(y + by + oy), vx = Math.trunc(x + bx + ox);
              if (vy >= 0 && vy < L && vx >= 0 && vx < C && img[vy][vx] < g[y][x]) {
                g[y][x] = img[vy][vx];
              }
            }
          }
        }
      }
      return g;
    }

    function labelK8(img) {
      var labels = Array.from({length: L}, function(){ return new Array(C).fill(0); });
      var dirs = [[-1, -1], [-1, 0], [-1, 1], [0, -1], [0, 1], [1, -1], [1, 0], [1, 1]];
      var cur = 0, desc = [];

      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) {
          if (img[y][x] === 1 && labels[y][x] === 0) {
            cur++;
            var stack = [[y, x]];
            labels[y][x] = cur;
            var area = 0, miny = y, maxy = y, minx = x, maxx = x;

            while (stack.length) {
              var cell = stack.pop();
              var cy = cell[0], cx = cell[1];
              area++;
              if (cy < miny) miny = cy;
              if (cy > maxy) maxy = cy;
              if (cx < minx) minx = cx;
              if (cx > maxx) maxx = cx;

              dirs.forEach(function(d){
                var ny = cy + d[0], nx = cx + d[1];
                if (ny >= 0 && ny < L && nx >= 0 && nx < C && img[ny][nx] === 1 && labels[ny][nx] === 0) {
                  labels[ny][nx] = cur;
                  stack.push([ny, nx]);
                }
              });
            }
            desc.push({k: cur, area: area, miny: miny, minx: minx, maxy: maxy, maxx: maxx});
          }
        }
      }
      return {labels: labels, desc: desc};
    }

    function paintBin(grid, img) {
      grid.innerHTML = '';
      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) {
          var c = document.createElement('div');
          c.style.cssText = 'width:26px;height:26px;border-radius:4px;border:1px solid #e4dcc8;box-sizing:border-box;';
          c.style.background = img[y][x] ? '#b9770e' : '#fafaf7';
          grid.appendChild(c);
        }
      }
    }

    function paintLabels(grid, labels) {
      grid.innerHTML = '';
      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) {
          var c = document.createElement('div');
          var k = labels[y][x];
          c.style.cssText = 'width:26px;height:26px;border-radius:4px;border:1px solid #e4dcc8;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;font-family:monospace;color:#ffffff;box-sizing:border-box;';
          c.style.background = k > 0 ? palette[(k - 1) % palette.length] : '#fafaf7';
          c.textContent = k > 0 ? k : '';
          grid.appendChild(c);
        }
      }
    }

    function render() {
      var p = parseInt(slP.value, 10) || 1;
      vlP.textContent = p + ' px';
      var f = buildF(p);
      var fe = erode(f, B);
      var res = labelK8(fe);

      paintBin(gF, f);
      paintLabels(gL, res.labels);

      var txt = '<b>' + res.desc.length + ' objeto(s) detectado(s) após a erosão.</b><br>';
      res.desc.forEach(function(d){
        txt += 'Rótulo ' + d.k + ': área = ' + d.area + ', bbox = (' + d.miny + ',' + d.minx + ') → (' + d.maxy + ',' + d.maxx + ')<br>';
      });
      if (res.desc.length < 2) {
        txt += '<i>A ponte ainda é espessa demais para a erosão 3×3 — as moedas continuam fundidas em 1 só objeto.</i>';
      }
      info.innerHTML = txt;
    }

    slP.addEventListener('input', render);

    render();
  }

  function tryInitSimEP0410(){
    var root = document.getElementById('sim-ep0410-rotulacao');
    if (root) initSimEP0410(root); else setTimeout(tryInitSimEP0410, 200);
  }
  tryInitSimEP0410();
})();
</script>
</div>
""")

**Figura 4.10:** Simulatore EP04_10: Separazione di Blob, Etichettatura e Descrittori


<figure id="fig-04-sim-ep0410-rotulacao">
  <img src="imagens/fig-04-sim-ep0410-rotulacao.png" alt=" Simulatore EP04_10: Separazione di Blob, Etichettatura e Descrittori " style="max-width:80%" />
  <figcaption><strong>Figura 4.10:</strong>  Simulatore EP04_10: Separazione di Blob, Etichettatura e Descrittori </figcaption>
</figure>

In [ ]:
%%writefile EP04_10.cpp
// your solution

In [ ]:
TestSuite("EP04_10.cpp").run()